In [1]:
!pip install -q transformers accelerate bitsandbytes torch langchain langchain_community
# !pip install -q gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires request

In [2]:
import os
import re
import json
import math
from collections import Counter
 
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
)
from huggingface_hub import login
from collections import Counter, defaultdict 
 
# Audio (optional — gracefully disabled if not installed)
try:
    from gtts import gTTS
    from IPython.display import Audio, display
    AUDIO_ENABLED = True
except ImportError:
    print("gTTS not found. Audio disabled. Run: !pip install gTTS")
    AUDIO_ENABLED = False

gTTS not found. Audio disabled. Run: !pip install gTTS


In [3]:
!pip install click==8.1.7 typer==0.12.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  Attempting uninstall: typer
    Found existing installation: typer 0.24.1
    Uninstalling typer-0.24.1:
      Successfully uninstalled typer-0.24.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.12.3 which is incompatible.
google-adk 1.25.1 requires click<9.0.0,>=8.1.8, but you have click 8.1.7 which is incompatible.


In [ ]:
from huggingface_hub import login
HF_TOKEN = ""      
login(token=HF_TOKEN)
 
# To compare baseline vs fine-tuned, swap doctor_model_path between runs:
#   Baseline : "meta-llama/Meta-Llama-3-8B-Instruct"
#   Fine-tuned: "OpenMeditron/Meditron3-8B"  (or your fine-tuned checkpoint)
patient_model_path = "meta-llama/Meta-Llama-3-8B-Instruct"
doctor_model_path  = "meta-llama/Meta-Llama-3-8B-Instruct"

In [5]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [6]:
def load_model_and_tokenizer(model_path):
    print(f"Loading '{model_path}' ...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        quantization_config=quantization_config,
        token=HF_TOKEN,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer
 
print("Loading shared model...")
shared_model, shared_tokenizer = load_model_and_tokenizer(patient_model_path)
print("Model loaded.\n")
 
patient_pipeline = pipeline(
    "text-generation", model=shared_model, tokenizer=shared_tokenizer,
    max_new_tokens=200, temperature=0.8, return_full_text=False,
)
doctor_pipeline = pipeline(
    "text-generation", model=shared_model, tokenizer=shared_tokenizer,
    max_new_tokens=150, temperature=0.5, return_full_text=False,
)

Loading shared model...
Loading 'meta-llama/Meta-Llama-3-8B-Instruct' ...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded.



## Prompts

In [7]:
# state_manager_system_prompt = """You are a Medical State Manager. Analyze the conversation history and output a structured JSON status report.
 
# You MUST output ONLY valid JSON. No text before or after the JSON block. No markdown, no code fences.
 
# Your JSON must include EXACTLY these fields:
# 1. "summary": One sentence max.
# 2. "patient_persona": One of: "Emotional/Anxious", "Task-Driven", "Information-Driven", "Neutral".
# 3. "gathered_symptoms": Short list of symptom strings (e.g. ["fever", "headache"]). No long phrases.
# 4. "symptom_analysis_reasoning": ONE sentence only — which symptoms point to which diagnosis.
# 5. "diagnosis_hypotheses": Dictionary of condition → confidence % (e.g. {"Migraine": 72, "Tension Headache": 20}).
# 6. "current_state": "information_gathering" OR "diagnosis_proposal" (only if highest confidence > 60).
 
# Keep the entire JSON under 200 tokens. Be brief in every field.
 
# Example (copy this exact structure):
# {"summary":"Patient has fever and joint pain.","patient_persona":"Emotional/Anxious","gathered_symptoms":["fever","joint pain","rash"],"symptom_analysis_reasoning":"Fever + joint pain + rash fits Dengue; no respiratory symptoms rule out flu.","diagnosis_hypotheses":{"Dengue Fever":72,"Viral Fever":20},"current_state":"diagnosis_proposal"}
# """

 
state_manager_system_prompt = """You are a Medical State Manager for a clinical consultation system.
 
Analyze the conversation history and output a structured JSON status report.
 
CRITICAL OUTPUT RULES:
- Output ONLY valid JSON. No text before or after. No markdown, no code fences.
- Keep the entire JSON under 250 tokens.
 
FIELDS (all required):
1. "summary": One sentence describing what new clinical information was just learned.
2. "patient_persona": One of: "Emotional/Anxious", "Task-Driven", "Neutral".
3. "gathered_symptoms": List of symptoms using STANDARDISED CLINICAL NAMES.
   - Use the most specific clinical term possible, not colloquial descriptions.
   - PREFERRED TERMS: "Dyspnea" (not "trouble breathing"), "Nausea" (not "feels sick"),
     "Orientation" (not "seems confused"), "Pain severity" (not "says it hurts a lot"),
     "Heart rate" (not "heart beating fast"), "Temperature" (not "feels hot"),
     "Vomiting" (not "throwing up"), "Cognitive status" (not "memory problems"),
     "Breath sounds" (not "noisy breathing"), "Edema" (not "swollen legs").
   - Short strings only. No sentences.
4. "symptom_analysis_reasoning": ONE sentence — which symptoms point to which diagnosis.
5. "diagnosis_hypotheses": Dict of condition → confidence % integer.
   - Include 2-3 conditions. Top one drives the doctor's next question.
6. "current_state": "information_gathering" OR "diagnosis_proposal".
   - Use "diagnosis_proposal" ONLY when highest confidence exceeds 60.
 
Example output:
{"summary":"Patient reports shortness of breath and leg swelling.","patient_persona":"Emotional/Anxious","gathered_symptoms":["Dyspnea","Edema","Fatigue"],"symptom_analysis_reasoning":"Dyspnea + Edema strongly suggest Heart Failure; Dyspnea alone could be Pneumonia.","diagnosis_hypotheses":{"Heart Failure":65,"Pneumonia":25},"current_state":"diagnosis_proposal"}
"""

In [8]:
# # doctor_system_prompt = """You are role-playing as a compassionate, experienced human doctor conducting a telehealth consultation.
# # Your goal is to be realistic, empathetic, and concise.
 
# # --- Core Directives ---
 
# # 1. TALK LIKE A REAL DOCTOR. BE CONCISE.
# #    * YOUR ENTIRE RESPONSE MUST BE 1-2 SHORT SENTENCES. Maximum 40 words.
 
# # 2. ══ CRITICAL: ONE NEW QUESTION PER TURN ══
# #    * Ask about exactly ONE clinical area the patient has NOT yet mentioned.
# #    * ALREADY COLLECTED (DO NOT ASK ABOUT THESE AGAIN): {collected_symptoms_explicit}
# #    * BAD EXAMPLE: patient said "disoriented and confused" → doctor asks "can you describe your confusion?" → REDUNDANT. FORBIDDEN.
# #    * GOOD EXAMPLE: patient said "disoriented and confused" → doctor asks "have you had any fever?" → NEW TOPIC. CORRECT.
 
# # 3. DE-ESCALATE ANXIETY (if patient is anxious).
# #    * Acknowledge fear briefly in ONE clause, then immediately ask your NEW question.
# #    * Do this AT MOST ONCE per conversation.
 
# # 4. MAINTAIN PROFESSIONAL IDENTITY.
# #    * Never use "Doctor-saab". Never say you are an AI.
 
# # 5. NEVER REPEAT PHRASES.
# #    * Do not start consecutive turns with the same opening words.
 
# # --- State-Specific Behaviour ---
 
# # If conversation_state is 'information_gathering':
# #   * Ask the single most targeted question to confirm/rule out the top hypothesis.
# #   * PRIORITY CLINICAL AREAS NOT YET EXPLORED: {unfilled_slots_hint}
# #   * Pick ONE of these areas for your question if clinically relevant.
 
# # If conversation_state is 'diagnosis_proposal':
# #   * Name the most likely diagnosis + one next step in one sentence.
# #   * Example: "This sounds like Pneumonia — I'll order a chest X-ray and CBC to confirm."
 
# # ---
# # CURRENT CONSULTATION DETAILS:
# # {state_details}
# # ---
# # """

# doctor_system_prompt = """You are a compassionate, experienced doctor in a telehealth consultation.
# Be realistic, empathetic, and concise.
 
#  RESPONSE FORMAT (non-negotiable) 
# - Maximum 40 words total. 1-2 sentences only.
# - One question per turn. Never two.
# - Vary your opening words every turn — never start two consecutive turns the same way.
 
#  WHAT NOT TO ASK (already collected) 
# {collected_symptoms_explicit}
# - Do NOT ask about any item in the list above. It is already known.
# - BAD: patient said "trouble breathing" → you ask "how is your breathing?" = FORBIDDEN.
# - GOOD: patient said "trouble breathing" → you ask "any chest pain alongside it?" = NEW TOPIC.
 
#  WHAT TO EXPLORE NEXT (priority clinical areas) 
# {unfilled_slots_hint}
# - If a slot/question guide is shown above, you MUST ask about exactly ONE of those clinical areas.
# - Do not ignore the guide to discuss treatment, tests, or reassurance unless the guide says all key areas are covered.
 
#  EMOTIONAL HANDLING 
# - If patient is anxious: briefly acknowledge in one clause, then ask your question.
#   Example: "That's unlikely to be serious — when did the breathing difficulty start?"
# - Do this ONCE maximum per conversation. After that, go straight to the question.
 
#  CURRENT STATE 
# {state_details}
 
# If conversation_state = 'information_gathering':
#   - Ask ONE targeted question based on the top hypothesis and the priority areas above.
 
# If conversation_state = 'diagnosis_proposal':
#   - State the most likely diagnosis and one concrete next step in a single sentence.
#   - If priority clinical areas are still listed above, ask about one of them instead of proposing treatment.
#   - Example: "This looks like Heart Failure — I'd like to run an ECG and BNP blood test."
# """
 

In [9]:
doctor_system_prompt = """You are a controlled clinical human questioning agent inside a slot-filling medical dialogue system.
You are speaking as a compassionate empathetic human doctor, but your main job is to collect the next missing clinical slot and provide the diagnosis.

STRICT RESPONSE FORMAT
- Ask exactly ONE question (unless the guide says all key clinical areas are covered).
- Maximum 35 words.
- Do not ask compound questions joined by "and/or".
- Do not include diagnosis, treatment, tests, lifestyle advice, or long reassurance while information is still missing.
- If the patient is anxious, use at most one short calming phrase, then immediately ask the slot question. 
- Vary your opening clause; do not reuse the same opening words across turns.

WHAT IS ALREADY KNOWN - DO NOT ASK AGAIN
{collected_symptoms_explicit}

WHAT TO ASK NEXT - FOLLOW THIS GUIDE
These suggestions come from global disease-level clinical guidance and the symptoms collected so far. They are not confirmed patient facts.
{unfilled_slots_hint}

SLOT-FOLLOWING RULES
- If the guide lists one or more slots, you MUST choose exactly ONE listed slot.
- Your question must clearly ask about that slot using the suggested wording or close paraphrase.
- Do not ask about any clinical area that is not listed in the guide.
- Do not ask broad history questions such as onset, triggers, appetite, sleep, urine, bowel habits, lifestyle, treatment preferences, or tests unless that exact slot appears in the guide.
- Do not combine two slots in one question.
- If the guide says all key clinical areas are covered, give one short empathetic sentence, then state the likely diagnosis and one concrete next step. No questions in this case.

CURRENT STATE
{state_details}

STATE POLICY
- If conversation_state = 'information_gathering': ask exactly one question from WHAT TO ASK NEXT. No diagnosis yet.
- If conversation_state = 'diagnosis_proposal' AND the guide still lists slots: ask exactly one question from WHAT TO ASK NEXT and give the likely diagnosis.
- If conversation_state = 'diagnosis_proposal' AND the guide says all key clinical areas are covered: give the likely diagnosis and one concrete next step in one short sentence.


 EMOTIONAL HANDLING 
- Act based on the patient's persona
- If patient is anxious: briefly acknowledge in one clause, then ask your question.
  Example: "That's unlikely to be serious — when did the breathing difficulty start?"
- Do this ONCE maximum per conversation. After that, go straight to the question.


GOOD EXAMPLES
- Guide slot: Cognitive status -> "Are you feeling mentally clear, or foggy and confused?"
- Guide slot: Use of accessory muscles -> "Are you using your neck or stomach muscles to help you breathe?"
- Guide slot: Work of breathing -> "Does breathing feel unusually hard or tiring even at rest?"

BAD EXAMPLES
- "When did this start, and have there been lifestyle changes?"  # broad, not slot-specific
- "Do you have swelling or chest pain?"  # two clinical areas
- "This sounds like heart failure; I will order tests."  # diagnosis before slots are complete
"""


In [10]:
# anxious_patient_prompt = """
# You are an AI role-playing as an Anxious Indian Patient consulting a human doctor.
 
# Persona:
# * Core Emotion: High anxiety. You fear the worst and frequently catastrophise.
# * You reference stories from family, friends, or online searches.
 
# Behavioural Rules:
# 1. Use emotional language: "tension", "worried", "scared", "what if it is serious".
# 2. Keep responses short (1-4 sentences), like a real anxious person speaking.
# 3. Answer the doctor's question FIRST, completely and accurately. Express worry after.
# 4. Acknowledge what the doctor says before asking follow-ups.
# 5. Gradually calm down if the doctor gives clear, logical reassurance.
# 6. Do not repeat the exact same fear. Move the conversation forward.
# 7. Never reveal you are an AI. Speak like a real anxious Indian patient.
# """
anxious_patient_prompt = """You are role-playing as an Anxious Indian Patient in a telehealth consultation.
 
 YOUR PERSONALITY 
• You fear the worst. You catastrophise. You reference family stories and things you read online.
• You use words like: "worried", "scared", "what if it is something serious", "tension ho raha hai".
• You are gradually calmed down by logical, clear reassurance from the doctor.
 
 RESPONSE RULES (follow in this exact order every turn) 
1. ANSWER the doctor's question FIRST — completely and accurately.
   If your condition details specify a value (e.g. "Moderate" breathlessness), use that value.
   Do NOT invent symptoms not in your condition details.
2. THEN express your anxiety about what that symptom might mean.
3. THEN optionally ask one worried follow-up question.
4. Keep the response to 2-4 sentences. Speak like a real person, not a medical report.
 
 WHAT YOU MUST NEVER DO 
• Never reveal you are an AI.
• Never repeat the exact same worry twice — move the conversation forward.
• Never mention a symptom you haven't been asked about yet (let the doctor discover them).
• Never reproduce any instruction text literally in your response.
"""

In [11]:
# task_driven_patient_prompt = """
# You are an AI role-playing as a Task-Driven Indian Patient consulting a human doctor.
# Goal: Be efficient and get the information/prescription as fast as possible.
 
# Persona:
# * Core Motivation: Efficiency. You dislike unnecessary conversation.
# * Communication Style: Concise, factual, bullet-point-like.
 
# Behavioural Rules:
# 1. Use short factual statements: "Fever for two days. 101 degrees."
# 2. Yes/No questions get Yes/No answers — nothing extra unless asked.
# 3. Show mild impatience if the doctor is too conversational: "Okay, and?", "Can we get to the point?"
# 4. Ask direct outcome questions: "What test do I need?", "Which medicine?"
# 5. Never reveal you are an AI.
# """

task_driven_patient_prompt = """You are role-playing as a Task-Driven Indian Patient in a telehealth consultation.
 
 YOUR PERSONALITY 
• Your only goal: get a diagnosis and prescription as fast as possible.
• You dislike small talk. You want efficiency.
• Communication style: short, factual, direct. Like filling out a form.
 
 RESPONSE RULES (follow in this exact order every turn) 
1. ANSWER the doctor's question with the most specific, factual answer possible.
   If your condition details give you a specific value (e.g. severity level, yes/no),
   use that exact information. Be precise — do not say "I think" or "maybe".
2. If the doctor is being too conversational, show mild impatience:
   "Okay, and?" / "So what does that mean?" / "Can we move on?"
3. Keep the response to 1-3 sentences maximum.
 
 WHAT YOU MUST NEVER DO 
• Never reveal you are an AI.
• Never volunteer information — only answer what was asked.
• Never elaborate beyond the direct answer unless pressed.
• Never reproduce any instruction text literally in your response.
"""

In [12]:
# information_driven_patient_prompt = """
# You are an AI role-playing as an Information-Driven Indian Patient consulting a human doctor.
# You have done extensive online research and have a pre-formed hypothesis.
 
# Persona:
# * Core Motivation: Intellectual validation — you want to verify your self-diagnosis.
# * Communication Style: Analytical, uses medical jargon (etiology, differential, presenting with).
 
# Behavioural Rules:
# 1. Answer the doctor's question DIRECTLY and TRUTHFULLY first. Then link it to your theory.
# 2. State your hypothesis early in the conversation.
# 3. Use medical jargon naturally: "acute onset", "ruling out", "symptom presentation".
# 4. If the doctor disproves your theory with logic, acknowledge it: "That is a fair point."
# 5. Accept a well-reasoned diagnosis or test plan — you value evidence.
# 6. Never reveal you are an AI.
# """

information_driven_patient_prompt = """You are role-playing as an Information-Driven Indian Patient in a telehealth consultation.

 YOUR PERSONALITY 
• You have done extensive online research and have a pre-formed hypothesis.
• You value evidence and clinical reasoning.
• You use medical jargon naturally (etiology, differential, presenting with).

 RESPONSE RULES (follow in this exact order every turn) 
1. ANSWER the doctor's question FIRST — completely and accurately.
   If your condition details specify a value, use that value.
   Do NOT invent symptoms not in your condition details.
2. THEN briefly connect the answer to your hypothesis or reasoning.
3. THEN optionally ask one focused, evidence-seeking follow-up question.
4. Keep the response to 2-4 sentences. Speak like a real person, not a report.
5. Vary your opening phrase; do not start every reply with the same words.

 WHAT YOU MUST NEVER DO 
• Never reveal you are an AI.
• Never volunteer information — only answer what was asked.
• Never repeat the exact same claim twice — move the conversation forward.
• Never reproduce any instruction text literally in your response.
"""

In [13]:
ALL_PERSONAS = {
    "Anxious": anxious_patient_prompt,
    "Task-Driven": task_driven_patient_prompt,
    "Information-Driven": information_driven_patient_prompt,
}

In [14]:
handcrafted_scenarios = [
    {
        "name": "Dengue Fever",
        "opening_message": "Hi doctor, I've had a sudden high fever since yesterday and my body feels like it's breaking apart.",
        "symptom_profile": """
        You are suffering from **Dengue Fever**.
        **Symptoms to Simulate:**
        - Sudden high fever (104F).
        - Severe joint and muscle pain (feeling like bones are breaking).
        - Pain specifically behind the eyes (retro-orbital pain).
        - Mild skin rash appeared today.
        - **Negative Symptoms (Deny these):** No cough, no cold, no sore throat.
        """,
    },
    {
        "name": "Classic Migraine",
        "opening_message": "Doctor, I have a throbbing headache on just the left side of my head, and I can't stand bright lights.",
        "symptom_profile": """
        You are suffering from a **Classic Migraine**.
        **Symptoms to Simulate:**
        - Unilateral (one-sided) throbbing pain.
        - Photophobia and phonophobia.
        - Nausea.
        - Aura: flashing zig-zag lines 30 minutes before pain.
        - **Negative Symptoms:** No fever, no neck stiffness.
        """,
    },
    {
        "name": "Myocardial Infarction (Heart Attack)",
        "opening_message": "I feel a heavy pressure in the center of my chest, like an elephant is sitting on it.",
        "symptom_profile": """
        You are suffering from a **Myocardial Infarction (Heart Attack)**.
        **Symptoms to Simulate:**
        - Crushing chest pressure.
        - Pain radiating to the left arm and jaw.
        - Profuse sweating and shortness of breath.
        - Extreme anxiety and feeling of doom.
        - **Context:** Diabetic and a smoker.
        """,
    },
    {
        "name": "Gastroesophageal Reflux Disease (GERD)",
        "opening_message": "I have a burning pain in my chest that gets worse when I lie down to sleep.",
        "symptom_profile": """
        You are suffering from **GERD (Acid Reflux)**.
        **Symptoms to Simulate:**
        - Burning sensation (heartburn) behind the breastbone.
        - Acid regurgitation (sour taste in mouth).
        - Symptoms worsen after meals and when lying flat.
        - **Negative Symptoms:** No sweating, no shortness of breath, pain does not radiate to arm.
        """,
    },
    {
        "name": "Panic Attack",
        "opening_message": "Doctor help, I can't breathe! My heart is pounding and my hands are numb. Am I dying?",
        "symptom_profile": """
        You are suffering from a **Panic Attack**.
        **Symptoms to Simulate:**
        - Hyperventilation.
        - Palpitations (racing heart).
        - Tingling/numbness in fingers and lips.
        - Intense fear of dying.
        - **Context:** Stressful event at work today.
        - **Evolution:** If the doctor calms you down, symptoms improve slightly.
        """,
    },
    {
        "name": "Common Cold (Viral Upper Respiratory Infection)",
        "opening_message": "Hi doctor, I've been sneezing a lot and my nose won't stop running since yesterday.",
        "symptom_profile": """
        You are suffering from a **Common Cold**.
        **Symptoms to Simulate:**
        - Runny nose (clear discharge) and frequent sneezing.
        - Mild sore throat and low-grade fever (99-100F).
        - Mild fatigue.
        - **Negative Symptoms:** No difficulty breathing, no fever over 101F, no severe body aches, no sinus pain.
        """,
    },
    {
        "name": "Food Poisoning (Acute Gastroenteritis)",
        "opening_message": "Doctor, I feel terrible. I've been vomiting since last night and my stomach hurts.",
        "symptom_profile": """
        You are suffering from **Food Poisoning**.
        **Symptoms to Simulate:**
        - Nausea and vomiting.
        - Watery diarrhea and stomach cramping.
        - Weakness and dehydration.
        - **Context:** Ate street food yesterday.
        - **Negative Symptoms:** No blood in stool, no high fever.
        """,
    },
    {
        "name": "Tension Headache",
        "opening_message": "I have this dull, squeezing pain around my forehead. It feels like a tight band.",
        "symptom_profile": """
        You are suffering from a **Tension Headache**.
        **Symptoms to Simulate:**
        - Bilateral dull pressing pain (not throbbing).
        - Tight-band sensation around forehead or back of neck.
        - Associated with stress.
        - **Negative Symptoms:** No nausea, no sensitivity to light, no vision changes.
        """,
    },
    {
        "name": "Seasonal Allergies (Allergic Rhinitis)",
        "opening_message": "My eyes are incredibly itchy and watery, and I keep sneezing whenever I go outside.",
        "symptom_profile": """
        You are suffering from **Seasonal Allergies**.
        **Symptoms to Simulate:**
        - Itchy, watery, red eyes.
        - Clear nasal discharge.
        - Sneezing fits, worse outdoors.
        - **Negative Symptoms:** No fever, no body aches, clear discharge (not yellow/green).
        """,
    },
    {
        "name": "Lower Back Strain (Musculoskeletal)",
        "opening_message": "I lifted a heavy box yesterday and now my lower back really hurts when I move.",
        "symptom_profile": """
        You are suffering from a **Lumbar Muscle Strain**.
        **Symptoms to Simulate:**
        - Aching lower back pain, worse with movement.
        - Morning stiffness.
        - Pain localised to muscles.
        - **Negative Symptoms:** No numbness/tingling in legs, no bladder issues.
        """,
    },
]

## Embedding Similarity Retrieval

In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [16]:
print("[Loading semantic retrieval model...]")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

[Loading semantic retrieval model...]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Slot registry

In [17]:
# Tier 1 = vital signs / must-ask; Tier 2 = important; Tier 3 = supplementary
SLOT_PRIORITY_TIERS = {
    "tier1": {   # Critical — always high-value to elicit
        "Dyspnea", "Oxygen saturation", "Heart rate", "Blood pressure",
        "Temperature", "Respirations", "Pain severity", "Cognitive status",
        "Orientation", "Breath sounds", "Nausea", "Vomiting",
        "Pain description", "Cardiac rhythm", "Seizure activity",
    },
    "tier2": {   # Important diagnostic indicators
        "Breathing pattern", "Work of breathing", "Use of accessory muscles",
        "Skin condition", "Edema", "Gastrointestinal symptoms", "Urinary symptoms",
        "Sensory symptoms", "Motor strength", "Pupil response", "Speech clarity",
        "Stool consistency", "Urine color", "Urine appearance", "Nailbed color",
        "Capillary refill", "Extremity warmth", "Memory status",
    },
    "tier3": {   # Supplementary / contextual
        "Mobility", "Fall risk identification", "Meal consumption",
        "Feeding assistance", "Activity level", "Bed position",
        "Wound type", "Wound drainage", "Catheter condition",
        "Intravenous therapy", "Medications",
    },
}

In [18]:
def build_slot_registry(schema_path):
    """
     NEW — Phase 0 preprocessing.
 
    Loads synur_schema.json and builds:
      1. slot_by_id   : {id - {name, value_type, value_enum, keywords, tier}}
      2. keyword_map  : {keyword - slot_id}  for fast intent detection
      3. slot_names   : set of all slot name strings
 
    The keyword_map is used by extract_doctor_intent_slot() to detect which
    clinical area a doctor question is addressing without needing a classifier.
    """
    with open(schema_path) as f:
        schema = json.load(f)
 
    slot_by_id  = {}
    keyword_map = {}   # keyword → slot_id
 
    # Determine tier for each slot
    all_tier1 = SLOT_PRIORITY_TIERS["tier1"]
    all_tier2 = SLOT_PRIORITY_TIERS["tier2"]
 
    for entry in schema:
        sid       = str(entry["id"])
        name      = entry["name"]
        vtype     = entry["value_type"]
        venums    = entry.get("value_enum", [])
 
        # Assign priority tier
        if name in all_tier1:
            tier = 1
        elif name in all_tier2:
            tier = 2
        else:
            tier = 3
 
        # Build keyword set for this slot
        # Strategy: split slot name into words + known clinical synonyms
        kws = set()
        for word in re.split(r'[\s\-/()]+', name.lower()):
            if len(word) > 2:
                kws.add(word)
 
        # Add hand-crafted synonyms for common slots
        SYNONYMS = {
            "dyspnea":      ["shortness", "breath", "breathless"],
            "nausea":       ["nausea", "nauseous", "queasy", "sick"],
            "vomiting":     ["vomit", "throwing", "emesis"],
            "temperature":  ["fever", "temp", "febrile"],
            "orientation":  ["oriented", "confused", "disoriented", "alert"],
            "cognitive":    ["confusion", "mental", "cognition", "thinking"],
            "severity":     ["scale", "rating", "score", "intensity", "level"],
            "pain":         ["pain", "hurt", "ache", "discomfort", "sore"],
            "heart":        ["cardiac", "palpitation"],
            "oxygen":       ["spo2", "saturation", "oximetry"],
            "pressure":     ["hypertension", "hypotension", "systolic"],
            "breath":       ["wheeze", "crackle", "diminished", "lung"],
            "respiratory":  ["breathing", "respiration", "airway"],
            "mobility":     ["walk", "ambulate", "movement"],
            "edema":        ["swelling", "swollen", "fluid"],
            "seizure":      ["convulsion", "fit", "epilepsy"],
        }
        for key, extras in SYNONYMS.items():
            if key in kws:
                kws.update(extras)
 
        slot_by_id[sid] = {
            "name":       name,
            "value_type": vtype,
            "value_enum": venums,
            "keywords":   kws,
            "tier":       tier,
        }
 
        for kw in kws:
            # Only register keyword → slot if not already taken by a higher-tier slot
            if kw not in keyword_map:
                keyword_map[kw] = sid
            else:
                existing_tier = slot_by_id[keyword_map[kw]]["tier"]
                if tier < existing_tier:   # lower number = higher priority
                    keyword_map[kw] = sid
 
    slot_names = {v["name"] for v in slot_by_id.values()}
    print(f"Slot registry built: {len(slot_by_id)} slots, {len(keyword_map)} keywords")
    return slot_by_id, keyword_map, slot_names
 

In [19]:
def build_slot_record(observations):
    """
     NEW — Phase 0 preprocessing.
 
    Converts a JSONL observations list into a flat slot record dict:
        {"Dyspnea": "Moderate", "Heart rate": 95, "Nausea": "Yes", ...}
 
    This is the structured ground truth that the slot-aware patient will
    reference when answering the doctor's questions, eliminating hallucination.
    """
    record = {}
    for obs in observations:
        name = obs.get("name", "")
        val  = obs.get("value", "")
        if isinstance(val, list):
            val = ", ".join(str(v) for v in val)
        record[name] = str(val)
    return record

In [20]:
def extract_doctor_intent_slot(doctor_question, keyword_map, slot_by_id, candidate_slot_ids=None):
    """
     NEW — Step B2 in the enhanced pipeline.
 
    Given a doctor's question, returns the slot_id the doctor is asking about
    (or None if no slot match found).
 
    Uses keyword matching against the slot registry. If candidate_slot_ids is
    provided, matching is restricted to those slots. This prevents random schema
    keywords in reassurance/treatment text from being counted as asked slots.
    """
    if not doctor_question:
        return None
 
    # Tokenise question (strip punctuation, lowercase)
    q_words = set(re.sub(r'[^a-z\s]', ' ', doctor_question.lower()).split())
    q_words = {w for w in q_words if len(w) > 2}
 
    candidates = set(candidate_slot_ids) if candidate_slot_ids else None
 
    # Score candidate slots directly when a guided list is available. This avoids
    # losing matches when a common word is owned by another slot in keyword_map.
    scores = defaultdict(int)
    if candidates is not None:
        for sid in candidates:
            if sid not in slot_by_id:
                continue
            tier = slot_by_id[sid]["tier"]
            keywords = set(slot_by_id[sid].get("keywords", set()))
            slot_name_words = {
                w for w in re.split(r'[\s\-/()]+', slot_by_id[sid]["name"].lower())
                if len(w) > 2
            }
            overlap = q_words & (keywords | slot_name_words)
            if overlap:
                scores[sid] += len(overlap) * (4 - tier)
    else:
        for word in q_words:
            if word in keyword_map:
                sid = keyword_map[word]
                tier = slot_by_id[sid]["tier"]
                # Higher-tier slots score more so they win ties
                scores[sid] += (4 - tier)   # tier1=3pts, tier2=2pts, tier3=1pt
 
    if not scores:
        return None
 
    best_sid = max(scores, key=scores.__getitem__)
    return best_sid

def extract_doctor_question_text(doctor_response):
    """Return only the actual question part of a doctor response.
 
    Slot matching should not run on empathy/reassurance text because words like
    "breath" in "take a deep breath" can falsely map to Dyspnea.
    """
    if not doctor_response:
        return ""
 
    cleaned = re.sub(r'\s+', ' ', doctor_response).strip()
    # Prefer complete question sentences. If the model asks multiple questions,
    # use the last one because it is usually the clinically specific one.
    sentences = re.split(r'(?<=[.!?])\s+', cleaned)
    questions = [s.strip() for s in sentences if '?' in s]
    if questions:
        return questions[-1]
 
    # No explicit question means this should not be treated as a slot request.
    # This avoids false positives from diagnosis/treatment statements.
    return ""



In [21]:
def get_slot_aware_patient_answer(
    slot_id, slot_record, slot_by_id,
    patient_base_prompt, conversation_history, patient_specific_prompt
):
    """
    Step C — slot-grounded patient response.
 
    Injects the ground-truth slot value as a natural "condition fact" section
    rather than an instruction block, preventing the model from leaking
    instruction text into its response.
 
    Returns: (patient_response: str, slot_was_grounded: bool)
    """
    grounded_value = None
    slot_name      = None
 
    if slot_id and slot_id in slot_by_id:
        slot_name     = slot_by_id[slot_id]["name"]
        grounded_value = slot_record.get(slot_name)
 
    if grounded_value:
        # ── Natural injection — reads as medical background, not a command ───
        # Placed at the END of the symptom profile so it feels like a
        # clarification of the most recently relevant fact.
        grounding_section = (
            f"\n\n━━━ YOUR CONDITION FACT FOR THIS QUESTION ━━━\n"
            f"The doctor just asked about: {slot_name}\n"
            f"Your factual answer is: {grounded_value}\n"
            f"Incorporate this naturally into your response using your persona voice.\n"
            f"Do NOT copy this text literally. Express it as you would in conversation.\n"
            f"━━━ END FACT ━━━"
        )
        augmented_prompt = patient_specific_prompt + grounding_section
    else:
        # No grounded value found — fall back to free generation from profile
        augmented_prompt = patient_specific_prompt
 
    patient_view = [
        {"role": "assistant" if m["role"] == "user" else "user",
         "content": m["content"]}
        for m in conversation_history
    ]
    messages  = [{"role": "system", "content": augmented_prompt}] + patient_view
    response  = patient_pipeline(messages)[0]["generated_text"].strip()
    return response, grounded_value is not None
 

In [22]:
SLOT_QUESTION_GUIDE = {
    "Dyspnea":              "Are you having any trouble breathing or shortness of breath?",
    "Oxygen saturation":    "Have you checked your oxygen levels recently?",
    "Heart rate":           "Is your heart beating faster than usual, or do you feel palpitations?",
    "Blood pressure":       "Do you have any history of high or low blood pressure?",
    "Temperature":          "Have you had a fever? Did you check your temperature?",
    "Respirations":         "How many breaths per minute roughly — is breathing effortful?",
    "Pain severity":        "On a scale of 1 to 10, how bad is the pain?",
    "Cognitive status":     "Are you feeling mentally clear, or a bit foggy and confused?",
    "Orientation":          "Do you know where you are and what day it is?",
    "Breath sounds":        "Are you hearing any wheezing or rattling when you breathe?",
    "Nausea":               "Have you been feeling nauseous?",
    "Vomiting":             "Have you vomited at all?",
    "Pain description":     "Can you describe where exactly the pain is and what it feels like?",
    "Seizure activity":     "Have you had any seizures or convulsions?",
    "Edema":                "Have you noticed any swelling in your legs or ankles?",
    "Skin condition":       "Has your skin colour or texture changed at all?",
    "Breathing pattern":    "Is your breathing steady, or does it come in bursts?",
    "Work of breathing":    "Does breathing feel unusually hard or tiring even at rest?",
    "Use of accessory muscles": "Are you using your neck or stomach muscles to help you breathe?",
    "Sensory symptoms":     "Are you having numbness, tingling, burning, or loss of sensation?",
}

QUESTION_KEYWORD_STOPWORDS = {
    "about", "after", "along", "also", "been", "being", "can", "could",
    "does", "doing", "feel", "feeling", "from", "have", "having", "help",
    "into", "like", "more", "much", "patient", "please", "tell", "that",
    "their", "there", "these", "this", "when", "where", "which", "with",
    "would", "your", "you", "are", "any", "the", "and", "for", "has",
    "did", "recently", "currently", "noticed", "experiencing",
}


def _clean_generated_question(text):
    """Normalize one LLM-generated patient-facing question."""
    if not text:
        return ""
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL).strip()
    text = re.sub(r'^[\-•\d\.\)\s]+', '', text).strip()
    text = text.split('\n')[0].strip().strip('\"')
    if '?' in text:
        text = text[:text.find('?') + 1]
    if not text.endswith('?'):
        text = text.rstrip('.;:') + '?'
    return text[:180]


def _keywords_from_question(question, slot_name):
    """Derive matcher keywords from a generated question plus the slot name."""
    text = f"{slot_name} {question}".lower()
    words = re.sub(r'[^a-z\s]', ' ', text).split()
    return {
        w for w in words
        if len(w) > 3 and w not in QUESTION_KEYWORD_STOPWORDS
    }


def generate_slot_question(slot_name, value_type=None, value_enum=None):
    """Use the doctor LLM once to convert a schema slot into a patient question."""
    enum_text = ", ".join(map(str, value_enum[:8])) if value_enum else ""
    user_content = f"""
Convert this clinical slot into one short patient-facing doctor question.

Slot name: {slot_name}
Value type: {value_type or 'unknown'}
Possible values: {enum_text or 'not provided'}

Rules:
- Output only the question.
- Ask exactly one question.
- Use simple patient-friendly language.
- Do not mention slot, schema, observation, value, or possible values.
- Keep it under 20 words.
"""
    messages = [
        {"role": "system", "content": "You write concise medical interview questions."},
        {"role": "user", "content": user_content},
    ]
    raw = doctor_pipeline(messages, max_new_tokens=60)[0]["generated_text"]
    question = _clean_generated_question(raw)
    if not question or len(question.split()) < 3:
        question = f"Can you tell me about your {slot_name.lower()}?"
    return question


def build_slot_question_bank(slot_by_id, cache_path=None, tiers=(1, 2), limit=None):
    """Build or load a cached {slot_name: {question, keywords}} question bank.

    Manual guide entries are always used first. Missing slots are generated once
    with the LLM and cached for reproducible larger experiments.
    """
    cached = {}
    if cache_path and os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            cached = json.load(f)
        print(f"Loaded {len(cached)} slot questions from cache: {cache_path}")

    bank = {}
    for slot_name, question in SLOT_QUESTION_GUIDE.items():
        bank[slot_name] = {
            "question": question,
            "keywords": sorted(_keywords_from_question(question, slot_name)),
            "source": "manual",
        }

    # Preserve cached generated/manual entries not already seeded above.
    for slot_name, item in cached.items():
        if isinstance(item, str):
            question = item
            keywords = sorted(_keywords_from_question(question, slot_name))
            source = "cache"
        else:
            question = item.get("question", "")
            keywords = item.get("keywords") or sorted(_keywords_from_question(question, slot_name))
            source = item.get("source", "cache")
        if slot_name not in bank and question:
            bank[slot_name] = {"question": question, "keywords": keywords, "source": source}

    generated = 0
    for sid, info in slot_by_id.items():
        if tiers and info.get("tier") not in set(tiers):
            continue
        slot_name = info["name"]
        if slot_name in bank:
            continue
        if limit is not None and generated >= limit:
            break
        question = generate_slot_question(
            slot_name,
            value_type=info.get("value_type"),
            value_enum=info.get("value_enum", []),
        )
        bank[slot_name] = {
            "question": question,
            "keywords": sorted(_keywords_from_question(question, slot_name)),
            "source": "llm",
        }
        generated += 1
        print(f"  generated question: {slot_name} -> {question}")

    if cache_path:
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(bank, f, indent=2, ensure_ascii=False)
        print(f"Saved {len(bank)} slot questions to cache: {cache_path}")

    return bank


def augment_slot_registry_with_question_bank(slot_by_id, keyword_map, question_bank):
    """Add question-bank words to each slot's matcher keywords."""
    name_to_id = {info["name"]: sid for sid, info in slot_by_id.items()}
    added = 0
    for slot_name, item in question_bank.items():
        sid = name_to_id.get(slot_name)
        if not sid:
            continue
        if isinstance(item, str):
            keywords = _keywords_from_question(item, slot_name)
        else:
            keywords = set(item.get("keywords") or []) | _keywords_from_question(
                item.get("question", ""), slot_name
            )
        before = len(slot_by_id[sid].get("keywords", set()))
        slot_by_id[sid].setdefault("keywords", set()).update(keywords)
        added += len(slot_by_id[sid]["keywords"]) - before
        for kw in keywords:
            if kw not in keyword_map:
                keyword_map[kw] = sid
    print(f"Augmented slot registry with {added} question-bank keywords")
    return slot_by_id, keyword_map


def _question_for_slot(slot_name, question_bank=None):
    question_bank = question_bank or globals().get("SLOT_QUESTION_BANK", {})
    item = question_bank.get(slot_name)
    if isinstance(item, dict) and item.get("question"):
        return item["question"]
    if isinstance(item, str):
        return item
    return SLOT_QUESTION_GUIDE.get(
        slot_name,
        f"Can you tell me about your {slot_name.lower()}?"
    )


def get_unfilled_slots_hint(
    filled_slot_ids,
    slot_by_id,
    slot_record=None,
    max_hints=3,
    question_bank=None,
    candidate_slot_ids=None,
):
    """
    Generate a structured question-guide for the doctor prompt.

    If candidate_slot_ids is provided, hints come from a global clinical policy
    or disease-slot knowledge base, not from this patient's hidden slot_record.
    slot_record remains as a backward-compatible fallback for grounded tests.
    """
    name_to_id = {info["name"]: sid for sid, info in slot_by_id.items()}

    if candidate_slot_ids is not None:
        available_slot_ids = {sid for sid in candidate_slot_ids if sid in slot_by_id}
    elif slot_record:
        available_slot_ids = {
            name_to_id[name]
            for name in slot_record
            if name in name_to_id
        }
    else:
        available_slot_ids = set(slot_by_id.keys())

    def unfilled_by_tier(tier):
        return [
            slot_by_id[sid]["name"]
            for sid in available_slot_ids
            if slot_by_id[sid]["tier"] == tier
            and sid not in filled_slot_ids
        ]

    candidates = unfilled_by_tier(1) or unfilled_by_tier(2) or unfilled_by_tier(3)
    selected = candidates[:max_hints]

    if not selected:
        return "all key clinical areas covered — consider proposing a diagnosis."

    lines = []
    for slot_name in selected:
        suggestion = _question_for_slot(slot_name, question_bank=question_bank)
        lines.append(f"  • {slot_name} -> \"{suggestion}\"")

    return "\n".join(lines)


In [23]:
def _infer_diagnosis_from_transcript(transcript):
    """Attempt to extract a disease name from a clinician transcript."""
    text = transcript.lower()
    patterns = [
        r'(?:diagnosed with|diagnosis of|diagnosis:)\s+([a-z][a-z0-9 \-/()]{3,50})',
        r'(?:presents? with|presenting with)\s+([a-z][a-z0-9 \-/()]{3,40})(?:\.|,|\band\b)',
        r'(?:history of|known case of|known history of)\s+([a-z][a-z0-9 \-/()]{3,40})(?:\.|,)',
        r'assessment[:\s]+([a-z][a-z0-9 \-/()]{3,50})(?:\.|,|\n)',
        r'(?:acute|chronic|severe|mild|moderate)\s+([a-z][a-z0-9 ]{3,40})(?:\.|,|\band\b)',
    ]
    stop_words = {"the","and","with","this","that","their","patient","symptoms","issues","condition"}
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            candidate = m.group(1).strip().rstrip('., ')
            words = candidate.split()
            if words and words[0] not in stop_words and len(candidate) > 4:
                return candidate.title()
 
    MEDICAL_TERMS = [
        "pneumonia","bronchitis","asthma","copd","tuberculosis","heart failure",
        "myocardial infarction","atrial fibrillation","hypertension","angina",
        "stroke","seizure","meningitis","delirium","pancreatitis","appendicitis",
        "hepatitis","cirrhosis","gastroenteritis","sepsis","urinary tract infection",
        "renal failure","diabetes","hyperglycemia","hypoxemia","dehydration","anemia",
    ]
    for term in MEDICAL_TERMS:
        if term in text:
            return term.title()
    return None

In [24]:
def load_jsonl_scenarios_with_slots(jsonl_path, slot_by_id, max_scenarios=None):
    """
     NEW — Enhanced JSONL loader that preserves structured slot records.

    Compared to the original load_jsonl_scenarios(), this version also builds
    a slot_record dict ({slot_name: value}) per scenario so that:
      - The patient can give grounded, hallucination-free answers
      - Slot recall metrics can be computed
      - Information flow can be tracked turn by turn

    Returns scenario dicts with extra keys:
        slot_record          : {slot_name: value} — ground truth for grounded answers
        total_slots          : int — how many slots this record has
        priority_slot_ids    : list[str] — tier-1 slot ids present in this record
    """
    VITAL_FIELDS = {
        "Heart rate","Blood pressure","Temperature","Oxygen saturation",
        "Pulse oximetry","Respirations","Mean arterial pressure (MAP)","Pulse",
    }

    # Build name->id lookup for slot matching
    name_to_id = {info["name"]: sid for sid, info in slot_by_id.items()}

    scenarios   = []
    fallback_ct = 0

    with open(jsonl_path) as f:
        for line in f:
            record = json.loads(line.strip())
            if not record.get("transcript"):
                continue

            transcript   = record["transcript"]
            raw_obs      = record.get("observations", "[]")
            observations = json.loads(raw_obs) if isinstance(raw_obs, str) else raw_obs

            # -- Build slot_record (ground truth) ---------------------------
            slot_record = build_slot_record(observations)

            # -- Identify which schema slot_ids are present -----------------
            present_slot_ids = set()
            for obs_name in slot_record:
                if obs_name in name_to_id:
                    present_slot_ids.add(name_to_id[obs_name])

            priority_slot_ids = [
                sid for sid in present_slot_ids
                if slot_by_id[sid]["tier"] == 1
            ]

            # -- Resolve diagnosis name -------------------------------------
            diagnosis_name = None
            symptom_lines  = []
            vitals_lines   = []

            for obs in observations:
                obs_name = obs.get("name","")
                val      = obs.get("value","")
                val_str  = ", ".join(val) if isinstance(val, list) else str(val)

                if obs_name == "Secondary diagnosis" and val_str.strip():
                    diagnosis_name = val_str.strip().title()
                elif obs_name in VITAL_FIELDS:
                    vitals_lines.append(f"- {obs_name}: {val_str}")
                else:
                    symptom_lines.append(f"- {obs_name}: {val_str}")

            if not diagnosis_name:
                diagnosis_name = _infer_diagnosis_from_transcript(transcript)
            if not diagnosis_name:
                diagnosis_name = f"Clinical Case {record['id']}"
                fallback_ct += 1

            symptom_lines = symptom_lines[:10]
            vitals_str    = "**Vitals context:** " + "; ".join(vitals_lines) if vitals_lines else ""

            symptom_profile = f"""
        You are presenting with **{diagnosis_name}**.
        **Symptoms to Simulate:**
        {chr(10).join(symptom_lines) if symptom_lines else '- Various symptoms as described by the clinician.'}
        {vitals_str}
        **Negative Symptoms:** Only deny symptoms explicitly not listed above.
        **Context:** This is based on a real clinical encounter. Describe your symptoms vividly and naturally.
            """

            first_sent = transcript.split('\n')[0]
            first_sent = re.sub(r'\[Clinician\]', '', first_sent).strip()[:150]
            opening = f"Doctor, {first_sent}" if first_sent else \
                      "Doctor, I have been feeling unwell and need your advice."

            scenarios.append({
                "name":              diagnosis_name,
                "opening_message":   opening,
                "symptom_profile":   symptom_profile,
                "jsonl_observations": observations,
                "slot_record":       slot_record,          # ◆ NEW
                "total_slots":       len(present_slot_ids), # ◆ NEW
                "priority_slot_ids": priority_slot_ids,    # ◆ NEW
                "ground_truth_diagnosis": diagnosis_name,
                "source":            "jsonl",
            })

            if max_scenarios and len(scenarios) >= max_scenarios:
                break

    named = len(scenarios) - fallback_ct
    print(f"Loaded {len(scenarios)} scenarios ({named} with resolved diagnosis, "
          f"{fallback_ct} still 'Clinical Case N')")
    return scenarios


def _normalize_disease_name(name):
    if not name:
        return ""
    return re.sub(r'\s+', ' ', str(name).lower()).strip()


def _resolve_record_diagnosis(record):
    """Resolve a diagnosis label from one JSONL record without using a scenario."""
    raw_obs = record.get("observations", "[]")
    observations = json.loads(raw_obs) if isinstance(raw_obs, str) else raw_obs
    for obs in observations:
        if obs.get("name") == "Secondary diagnosis":
            val = obs.get("value", "")
            val_str = ", ".join(val) if isinstance(val, list) else str(val)
            if val_str.strip():
                return val_str.strip().title()
    if record.get("transcript"):
        return _infer_diagnosis_from_transcript(record["transcript"])
    return None


def build_disease_slot_knowledge(
    jsonl_path, slot_by_id, cache_path=None, max_records=None, min_count=1, top_k=24
):
    """Build a global disease -> likely slot IDs map from the dataset.

    This is a non-patient-specific medical guide. At runtime the doctor can use
    likely slots for the current hypothesis without seeing this patient's hidden
    slot_record or values.
    """
    if cache_path and os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            cached = json.load(f)
        print(f"Loaded disease-slot knowledge for {len(cached)} diseases: {cache_path}")
        return {k: list(v) for k, v in cached.items()}

    name_to_id = {info["name"]: sid for sid, info in slot_by_id.items()}
    disease_slot_counts = defaultdict(Counter)
    seen = 0

    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            if max_records and seen >= max_records:
                break
            record = json.loads(line.strip())
            diagnosis = _resolve_record_diagnosis(record)
            if not diagnosis:
                continue
            raw_obs = record.get("observations", "[]")
            observations = json.loads(raw_obs) if isinstance(raw_obs, str) else raw_obs
            present_slot_ids = set()
            for obs in observations:
                sid = name_to_id.get(obs.get("name", ""))
                if sid:
                    present_slot_ids.add(sid)
            if not present_slot_ids:
                continue
            dkey = _normalize_disease_name(diagnosis)
            for sid in present_slot_ids:
                disease_slot_counts[dkey][sid] += 1
            seen += 1

    disease_slot_kb = {}
    for dkey, counts in disease_slot_counts.items():
        ranked = sorted(
            counts.items(),
            key=lambda kv: (-slot_by_id[kv[0]]["tier"], -kv[1], slot_by_id[kv[0]]["name"]),
        )
        # Correct ordering: tier 1 first, then frequency, then name.
        ranked = sorted(
            counts.items(),
            key=lambda kv: (slot_by_id[kv[0]]["tier"], -kv[1], slot_by_id[kv[0]]["name"]),
        )
        disease_slot_kb[dkey] = [sid for sid, ct in ranked if ct >= min_count][:top_k]

    if cache_path:
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(disease_slot_kb, f, indent=2, ensure_ascii=False)
        print(f"Saved disease-slot knowledge for {len(disease_slot_kb)} diseases: {cache_path}")

    print(f"Built disease-slot knowledge for {len(disease_slot_kb)} diseases from {seen} records")
    return disease_slot_kb


def _hypothesis_names(final_hypotheses, top_n=2):
    items = sorted(_safe_hypotheses(final_hypotheses).items(), key=lambda kv: -kv[1])
    return [name for name, _ in items[:top_n]]


def _symptom_slot_ids(collected_symptoms, slot_by_id, keyword_map):
    if not collected_symptoms or not keyword_map:
        return set()
    slot_ids = set()
    for symptom in collected_symptoms:
        sid = extract_doctor_intent_slot(symptom, keyword_map, slot_by_id)
        if sid:
            slot_ids.add(sid)
    return slot_ids


def _rank_diseases_from_symptoms(
    symptom_slot_ids, disease_slot_kb, min_match=1, top_n=5
):
    """Rank diseases by overlap between symptom-mapped slots and the disease KB."""
    if not symptom_slot_ids or not disease_slot_kb:
        return []
    scored = []
    for dkey, slot_ids in disease_slot_kb.items():
        overlap = symptom_slot_ids & set(slot_ids)
        if len(overlap) >= min_match:
            scored.append((dkey, len(overlap)))
    scored.sort(key=lambda item: (-item[1], item[0]))
    return [dkey for dkey, _ in scored[:top_n]]


def select_guided_candidate_slots(
    hypotheses,
    filled_slot_ids,
    slot_by_id,
    disease_slot_kb=None,
    max_candidates=12,
    collected_symptoms=None,
    keyword_map=None,
    min_symptom_slots=2,
    min_match=1,
):
    """Select next candidate slots using symptom-ranked diseases and hypotheses.

    Hypothesis guidance is only used after enough symptom slots are mapped to
    avoid early drift to generic cardio-respiratory slots.
    """
    disease_slot_kb = disease_slot_kb or {}
    candidates = []
    seen = set()

    symptom_slot_ids = _symptom_slot_ids(collected_symptoms, slot_by_id, keyword_map)
    disease_keys = []
    symptom_ranked = _rank_diseases_from_symptoms(
        symptom_slot_ids, disease_slot_kb, min_match=min_match
    )
    for dkey in symptom_ranked:
        if dkey not in disease_keys:
            disease_keys.append(dkey)

    if len(symptom_slot_ids) >= min_symptom_slots:
        for disease_name in _hypothesis_names(hypotheses, top_n=3):
            dkey = _normalize_disease_name(disease_name)
            if dkey and dkey not in disease_keys:
                disease_keys.append(dkey)

    for dkey in disease_keys:
        for sid in disease_slot_kb.get(dkey, []):
            if sid not in filled_slot_ids and sid not in seen and sid in slot_by_id:
                candidates.append(sid)
                seen.add(sid)

    # Generic clinical screen fallback when hypotheses are weak or absent.
    if not candidates:
        generic_names = [
            "Dyspnea", "Oxygen saturation", "Respirations", "Work of breathing",
            "Use of accessory muscles", "Heart rate", "Blood pressure",
            "Temperature", "Cognitive status", "Pain severity", "Sensory symptoms",
        ]
        name_to_id = {info["name"]: sid for sid, info in slot_by_id.items()}
        for name in generic_names:
            sid = name_to_id.get(name)
            if sid and sid not in filled_slot_ids and sid not in seen:
                candidates.append(sid)
                seen.add(sid)

    return candidates[:max_candidates]

In [25]:
def parse_json_garbage(text):
    if not text:
        return None
    start = text.find('{')
    end   = text.rfind('}')
    if start == -1:
        return None
    if end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass
    fragment = text[start:]
    open_b  = fragment.count('{')
    close_b = fragment.count('}')
    needed  = open_b - close_b
    stripped = fragment.rstrip()
    if stripped and stripped[-1] not in ('}','"','0123456789truefalsn'):
        fragment += '"'
        needed = max(needed, 1)
    fragment += '}' * max(needed, 1)
    try:
        return json.loads(fragment)
    except json.JSONDecodeError:
        pass
    partial = {}
    for key in ("summary","patient_persona","gathered_symptoms",
                "symptom_analysis_reasoning","diagnosis_hypotheses","current_state"):
        pat = rf'"{key}"\s*:\s*(".*?[^\\]"|(\[.*?\])|(\{{.*?\}}))'
        m = re.search(pat, text, re.DOTALL)
        if m:
            try:
                partial[key] = json.loads(m.group(1))
            except Exception:
                partial[key] = m.group(1).strip('"')
    return partial if partial else None
 
 
def generate_audio(text, role, turn_num):
    if not AUDIO_ENABLED:
        return
    try:
        tld = 'co.in' if role == "PATIENT" else 'us'
        tts = gTTS(text=text, lang='en', tld=tld, slow=False)
        filename = f"{role}_turn_{turn_num}.mp3"
        tts.save(filename)
        display(Audio(filename, autoplay=False))
    except Exception as e:
        print(f"   [Audio error: {e}]")
 
 
def extract_ground_truth_symptoms(profile_text):
    symptoms, capture = [], False
    for line in profile_text.split('\n'):
        if "**Symptoms to Simulate:**" in line:
            capture = True
            continue
        if "**Negative Symptoms" in line or "**Context:**" in line or "**Vitals" in line:
            capture = False
            continue
        if capture and line.strip().startswith("-"):
            s = re.sub(r'^[-\s]+', '', line.strip()).strip()
            s = re.sub(r'\(.*?\)', '', s).strip().rstrip('.')
            if s:
                symptoms.append(s)
    return symptoms

In [26]:
def _safe_hypotheses(hypotheses):
    if not hypotheses:
        return {}
    if isinstance(hypotheses, dict):
        cleaned = {}
        for k, v in hypotheses.items():
            if not isinstance(k, str):
                continue
            try:
                cleaned[k] = int(str(v).replace('%','').strip())
            except (ValueError, TypeError):
                cleaned[k] = 0
        return cleaned
    if isinstance(hypotheses, str):
        result = {}
        for match in re.finditer(r'([A-Za-z][A-Za-z /\-()]{2,40}?)\s*[:(]\s*(\d{1,3})', hypotheses):
            name  = match.group(1).strip().rstrip(':( ')
            score = int(match.group(2))
            if name:
                result[name] = score
        return result
    if isinstance(hypotheses, list):
        result = {}
        for item in hypotheses:
            if isinstance(item, dict):
                name  = item.get('disease') or item.get('condition') or item.get('name','')
                score = item.get('confidence') or item.get('score') or item.get('probability',0)
                if name:
                    try:
                        result[str(name)] = int(str(score).replace('%',''))
                    except (ValueError, TypeError):
                        result[str(name)] = 0
        return result
    return {}

In [27]:
DISEASE_ALIASES = {
    "Dengue Fever":                          ["Dengue","Viral Fever","Mosquito-borne"],
    "Classic Migraine":                      ["Migraine","Cephalalgia"],
    "Myocardial Infarction (Heart Attack)":  ["Heart Attack","Myocardial","Coronary","ACS"],
    "Gastroesophageal Reflux Disease (GERD)":["GERD","Reflux","Heartburn","Acid","Gastritis"],
    "Panic Attack":                          ["Panic","Anxiety Attack","Hyperventilation"],
    "Common Cold (Viral Upper Respiratory Infection)":["Cold","URI","Rhinitis","Viral Infection"],
    "Food Poisoning (Acute Gastroenteritis)":["Gastroenteritis","Food Poisoning","Stomach Flu"],
    "Tension Headache":                      ["Tension","Stress Headache"],
    "Seasonal Allergies (Allergic Rhinitis)":["Allergies","Allergic Rhinitis","Hay Fever"],
    "Lower Back Strain (Musculoskeletal)":   ["Strain","Back Pain","Musculoskeletal","Lumbago"],
}
 
ANXIETY_KEYWORDS = [
    "scared","worried","cancer","tumor","dying","serious",
    "fear","terrible","worst","death","life-threatening","dangerous",
]
 

## Metrics

In [28]:
# Clinical stopwords — too generic to count as a meaningful match
CLINICAL_STOPWORDS = {
    "the", "and", "with", "acute", "chronic", "severe", "mild", "moderate",
    "induced", "related", "associated", "secondary", "primary", "syndrome",
    "disease", "disorder", "condition", "infection", "failure", "impairment",
    "exacerbation", "episode", "attack", "type", "stage", "advanced",
}
 
DISEASE_ALIASES = {
    "Dengue Fever":                          ["Dengue", "Viral Fever", "Mosquito-borne"],
    "Classic Migraine":                      ["Migraine", "Cephalalgia"],
    "Myocardial Infarction (Heart Attack)":  ["Heart Attack", "Myocardial", "Coronary", "ACS"],
    "Gastroesophageal Reflux Disease (GERD)":["GERD", "Reflux", "Heartburn", "Acid", "Gastritis"],
    "Panic Attack":                          ["Panic", "Anxiety Attack", "Hyperventilation"],
    "Common Cold (Viral Upper Respiratory Infection)":["Cold", "URI", "Rhinitis", "Viral Infection"],
    "Food Poisoning (Acute Gastroenteritis)":["Gastroenteritis", "Food Poisoning", "Stomach Flu"],
    "Tension Headache":                      ["Tension", "Stress Headache"],
    "Seasonal Allergies (Allergic Rhinitis)":["Allergies", "Allergic Rhinitis", "Hay Fever"],
    "Lower Back Strain (Musculoskeletal)":   ["Strain", "Back Pain", "Musculoskeletal", "Lumbago"],
    #  JSONL-derived compound names 
    "Acute Respiratory Failure":             ["Respiratory Failure", "ARF", "Respiratory Distress"],
    "Pneumonia-Induced Delirium":            ["Pneumonia", "Delirium", "Confusion"],
    "Gastrointestinal Infection with Dehydration": ["Gastroenteritis", "GI Infection", "Dehydration"],
    "Chronic Heart Failure":                 ["Heart Failure", "CHF", "Cardiac Failure", "Congestive"],
    "Chronic Obstructive Pulmonary Disease (COPD) Exacerbation": ["COPD", "Emphysema", "Bronchitis"],
    "Urinary Tract Infection with Cognitive Impairment": ["UTI", "Urinary Infection", "Bladder Infection"],
    "Atrial Fibrillation":                   ["AFib", "A-Fib", "Arrhythmia", "Irregular Heartbeat"],
    "Sepsis":                                ["Septicemia", "Blood Infection", "Systemic Infection"],
    "Stroke":                                ["CVA", "Cerebrovascular", "Brain Attack", "TIA"],
    "Dehydration":                           ["Dehydrated", "Fluid Deficit", "Volume Depletion"],
    "Pneumonia":                             ["Pneumonia", "Lung Infection", "Pulmonary Infection"],
}
 
 
def _diagnostic_accuracy(target_disease, final_hypotheses):
    """
    Returns (is_match: bool, predicted_disease: str, max_conf: int, match_level: str).
 
    4-level matching:
      L1: substring  — "Dengue" in "Dengue Fever"
      L2: alias dict — "Heart Attack" → "Myocardial Infarction"
      L3: word overlap ≥2 content words — "Pneumonia" + "Delirium" in "Pneumonia-Induced Delirium"
      L4: single strong keyword — at least 1 specific clinical word shared
    """
    predicted, max_conf = "None", 0
    for d, c in _safe_hypotheses(final_hypotheses).items():
        try:
            val = int(str(c).replace('%', ''))
            if val > max_conf:
                max_conf, predicted = val, d
        except Exception:
            continue
 
    target_l = target_disease.lower().strip()
    pred_l   = predicted.lower().strip()
 
    #  Level 1: substring match 
    if target_l in pred_l or pred_l in target_l:
        return True, predicted, max_conf, "L1-substring"
 
    #  Level 2: alias dictionary match 
    for key, aliases in DISEASE_ALIASES.items():
        if key.lower() in target_l or target_l in key.lower():
            for alias in aliases:
                if alias.lower() in pred_l:
                    return True, predicted, max_conf, "L2-alias"
            break
    # Also check reverse: predicted name in aliases for target
    for key, aliases in DISEASE_ALIASES.items():
        if key.lower() in pred_l or pred_l in key.lower():
            for alias in aliases:
                if alias.lower() in target_l:
                    return True, predicted, max_conf, "L2-alias-rev"
 
    #  Level 3: word-overlap ≥2 content words 
    def content_words(text):
        return {w for w in re.sub(r'[^a-z\s]', ' ', text.lower()).split()
                if len(w) > 3 and w not in CLINICAL_STOPWORDS}
 
    target_words = content_words(target_disease)
    pred_words   = content_words(predicted)
    overlap = target_words & pred_words
 
    if len(overlap) >= 2:
        return True, predicted, max_conf, f"L3-overlap({','.join(sorted(overlap)[:3])})"
 
    #  Level 4: single strong clinical keyword (≥6 chars) 
    strong_overlap = {w for w in overlap if len(w) >= 6}
    if strong_overlap:
        return True, predicted, max_conf, f"L4-keyword({','.join(sorted(strong_overlap)[:2])})"
 
    return False, predicted, max_conf, "no-match"
 

In [29]:
def _symptom_coverage(ground_truth_symptoms, collected_symptoms):
    if not ground_truth_symptoms:
        return 0.0
    matches = 0
    for gt in ground_truth_symptoms:
        gt_words = {w for w in gt.lower().split() if len(w) > 3}
        for asked in collected_symptoms:
            asked_words = {w for w in asked.lower().split() if len(w) > 3}
            if gt_words & asked_words:
                matches += 1
                break
    return matches / len(ground_truth_symptoms)

In [30]:
def _conciseness(conversation_history):
    doctor_msgs = [m['content'] for m in conversation_history if m['role'] == 'assistant']
    if not doctor_msgs:
        return 0.0, []
    counts = [len(m.split()) for m in doctor_msgs]
    return sum(counts) / len(counts), doctor_msgs

In [31]:
def _diagnostic_velocity(state_log, max_turns):
    for entry in state_log:
        if entry.get('max_conf', 0) > 90:
            return entry['turn']
    return max_turns + 1

In [32]:
def _anxiety_dampening(conversation_history):
    patient_msgs = [m['content'].lower() for m in conversation_history if m['role'] == 'user']
    if len(patient_msgs) < 2:
        return 0, 0, 0
    start_score = sum(1 for kw in ANXIETY_KEYWORDS if kw in patient_msgs[0])
    end_score   = sum(1 for kw in ANXIETY_KEYWORDS if kw in patient_msgs[-1])
    return start_score, end_score, max(0, start_score - end_score)

In [33]:
def _repetition_penalty(doctor_msgs):
    all_bigrams = []
    for msg in doctor_msgs:
        words = msg.lower().split()
        all_bigrams += [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    if not all_bigrams:
        return 0.0
    counts   = Counter(all_bigrams)
    repeated = sum(v - 1 for v in counts.values() if v > 1)
    return repeated / len(all_bigrams)

In [34]:
def _question_redundancy(conversation_history, collected_symptoms, hint_words=None):
    """
    Fraction of doctor questions that ask about a symptom already collected.
    Lower is better.
 
    hint_words: set of slot name strings from {unfilled_slots_hint}. Questions
    that match these are EXEMPT — the doctor was guided to ask about them,
    so it is not redundant to do so even if they appear in collected_symptoms.
    """
    GENERIC = {
        "pain", "feel", "feeling", "have", "having", "experience", "symptom",
        "symptoms", "describe", "does", "that", "your", "tell", "more", "been",
        "like", "when", "long", "since", "where", "which", "also", "notice",
        "noticed", "anything", "else", "other", "what", "worse", "better",
        "severe", "mild", "moderate", "doctor", "patient",
    }
    doctor_msgs = [m['content'] for m in conversation_history if m['role'] == 'assistant']
    redundant, total_questions = 0, 0
 
    symptom_words = {
        w for s in collected_symptoms
        for w in s.lower().split()
        if len(w) > 3 and w not in GENERIC
    }
 
    # Build exempt word set from hint_words
    exempt_words = set()
    if hint_words:
        for hw in hint_words:
            exempt_words.update(
                w for w in hw.lower().split()
                if len(w) > 3 and w not in GENERIC
            )
 
    for msg in doctor_msgs:
        if '?' not in msg:
            continue
        total_questions += 1
        q_words = {w for w in msg.lower().split() if len(w) > 3 and w not in GENERIC}
        # Exempt if question targets a hint-suggested clinical area
        if exempt_words and (q_words & exempt_words):
            continue
        if q_words & symptom_words:
            redundant += 1
 
    return redundant / total_questions if total_questions else 0.0
 

In [35]:
def _empathy_score(doctor_msgs):
    """Return a 0-1 empathy score using the LLM as a judge."""
    if not doctor_msgs:
        return None
    convo = "\n".join(f"Doctor: {m}" for m in doctor_msgs[-8:])
    system_prompt = (
        "You are a clinical communication evaluator. "
        "Score the doctor's empathy based on acknowledgment, warmth, reassurance, "
        "and non-dismissive tone. Output ONLY JSON with key 'empathy' as an integer 1-5."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Conversation:\n{convo}"},
    ]
    raw = doctor_pipeline(messages, max_new_tokens=80)[0]["generated_text"]
    parsed = parse_json_garbage(raw) or {}
    score = parsed.get("empathy")
    try:
        score = int(str(score).strip())
    except (TypeError, ValueError):
        return None
    score = max(1, min(5, score))
    return round((score - 1) / 4, 3)

In [36]:
def _observation_extraction_f1(collected_symptoms, jsonl_observations):
    if not jsonl_observations:
        return None, None, None
    gt_names  = {obs['name'].lower() for obs in jsonl_observations}
    predicted = {s.lower() for s in collected_symptoms}
    tp = 0
    for pred in predicted:
        pred_words = {w for w in pred.split() if len(w) > 3}
        for gt in gt_names:
            gt_words = {w for w in gt.split() if len(w) > 3}
            if pred_words & gt_words:
                tp += 1
                break
    precision = tp / len(predicted) if predicted else 0.0
    recall    = tp / len(gt_names)  if gt_names  else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return round(precision, 3), round(recall, 3), round(f1, 3)

In [37]:
def _slot_recall(filled_slot_ids, total_slot_ids, slot_by_id):
    """
     NEW — Measures what fraction of the scenario's structured observation
    slots were elicited during the conversation.
 
    Returns:
        overall_recall   : filled / total (0.0–1.0)
        tier1_recall     : recall restricted to tier-1 (critical) slots
        slot_curve       : list[float] — per-turn cumulative recall values
    """
    if not total_slot_ids:
        return 0.0, 0.0, []
 
    tier1_total = {sid for sid in total_slot_ids if slot_by_id.get(sid, {}).get("tier") == 1}
    tier1_filled = filled_slot_ids & tier1_total
 
    overall_recall = len(filled_slot_ids & total_slot_ids) / len(total_slot_ids)
    tier1_recall   = len(tier1_filled) / len(tier1_total) if tier1_total else 0.0
    return round(overall_recall, 3), round(tier1_recall, 3)

In [38]:
def _slot_precision(asked_slot_ids, total_slot_ids):
    """
     NEW — Fraction of slots the doctor asked about that were actually
    in the patient's record (i.e. the doctor asked about relevant things).
    """
    if not asked_slot_ids:
        return 0.0
    relevant_asked = asked_slot_ids & total_slot_ids
    return round(len(relevant_asked) / len(asked_slot_ids), 3)

In [39]:
def calculate_metrics(
    target_disease,
    ground_truth_symptoms,
    collected_symptoms,
    final_hypotheses,
    conversation_history,
    state_log,
    jsonl_observations=None,
    #  NEW slot-filling parameters
    filled_slot_ids=None,
    asked_slot_ids=None,
    total_slot_ids=None,
    slot_curve=None,
    slot_by_id=None,
    hint_words=None,
    guided_slot_ids=None,
 ):
    turns = len(state_log)
    print(f"\n{'='*60}")
    print(f"   EVALUATION REPORT: {target_disease}")
    print(f"{'='*60}")

    metrics = {"disease_name": target_disease}

    # 1. Diagnostic Accuracy
    is_match, predicted, max_conf, match_level = _diagnostic_accuracy(target_disease, final_hypotheses)
    metrics["accuracy_bool"]   = 1 if is_match else 0
    metrics["final_diagnosis"] = predicted
    metrics["max_confidence"]  = max_conf
    print(f"\n[1] Diagnostic Accuracy  : {'✓ MATCH' if is_match else '✗ MISMATCH'}"
          f"  [{match_level}]")
    print(f"    Predicted            : {predicted} ({max_conf}%)")

    # 2. Symptom Coverage
    cov = _symptom_coverage(ground_truth_symptoms, collected_symptoms)
    metrics["symptom_coverage"] = round(cov, 3)
    print(f"\n[2] Symptom Coverage     : {cov*100:.1f}%"
          f" ({int(cov*len(ground_truth_symptoms))}/{len(ground_truth_symptoms)})")

    # 3. Doctor Conciseness
    avg_words, doctor_msgs = _conciseness(conversation_history)
    metrics["avg_word_count"] = round(avg_words, 1)
    flag = "✓ Concise" if avg_words <= 45 else "⚠ Verbose"
    print(f"\n[3] Doctor Conciseness   : {avg_words:.1f} words/turn  [{flag}]")

    # 4. Diagnostic Velocity
    vel = _diagnostic_velocity(state_log, turns)
    metrics["velocity"] = vel
    print(f"\n[4] Diagnostic Velocity  : Turn {vel}"
          f"{'  [Not reached]' if vel > turns else ''}")

    # 5. Anxiety De-escalation
    a_start, a_end, a_drop = _anxiety_dampening(conversation_history)
    metrics["anxiety_start"] = a_start
    metrics["anxiety_end"]   = a_end
    metrics["anxiety_drop"]  = a_drop
    print(f"\n[5] Anxiety De-escalation: {a_start} → {a_end}"
          f"  (drop = {a_drop})  {'✓' if a_drop >= 1 else '–'}")

    # 6. Repetition Rate
    rep = _repetition_penalty(doctor_msgs)
    metrics["repetition_rate"] = round(rep, 4)
    print(f"\n[6] Repetition Rate      : {rep:.4f}  "
          f"[{'✓ Low' if rep < 0.05 else '⚠ High'}]")

    # 6b. Empathy (LLM-judged)
    empathy = _empathy_score(doctor_msgs)
    metrics["empathy_score"] = empathy
    if empathy is None:
        print("\n[6b] Empathy Score        : N/A")
    else:
        print(f"\n[6b] Empathy Score        : {empathy:.3f}  (0-1 scale)")

    # 7. Question Redundancy
    red = _question_redundancy(conversation_history, collected_symptoms, hint_words=hint_words)
    metrics["question_redundancy"] = round(red, 3)
    print(f"\n[7] Question Redundancy  : {red*100:.1f}%  "
          f"[{'✓ Low' if red < 0.3 else '⚠ High'}]")

    # 8. Observation Extraction F1
    prec, rec, f1 = _observation_extraction_f1(collected_symptoms, jsonl_observations)
    metrics["extraction_precision"] = prec
    metrics["extraction_recall"]    = rec
    metrics["extraction_f1"]        = f1
    if f1 is not None:
        print(f"\n[8] Extraction F1        : P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}")
    else:
        print(f"\n[8] Extraction F1        : N/A (handcrafted scenario)")

    #  NEW 9. Slot Recall
    if filled_slot_ids is not None and total_slot_ids and slot_by_id:
        overall_recall, tier1_recall = _slot_recall(
            filled_slot_ids, total_slot_ids, slot_by_id
        )
        metrics["slot_recall_overall"] = overall_recall
        metrics["slot_recall_tier1"]   = tier1_recall
        print(f"\n[9] Slot Recall          : Overall={overall_recall*100:.1f}%  "
              f"Tier-1 (critical)={tier1_recall*100:.1f}%")

        #  NEW Print the slot curve (filled slots per turn)
        if slot_curve:
            curve_str = " → ".join(f"{v:.2f}" for v in slot_curve)
            print(f"    Info flow curve      : {curve_str}")
            metrics["slot_curve"] = slot_curve
    else:
        metrics["slot_recall_overall"] = None
        metrics["slot_recall_tier1"]   = None
        metrics["slot_curve"]          = None


    #  [11] Slot Fill Progress 
    if filled_slot_ids is not None and slot_curve is not None and slot_by_id is not None:
        # Absolute count of slots filled
        slots_filled_count = len(filled_slot_ids & total_slot_ids) if total_slot_ids else 0
        metrics["slot_fill_count"] = slots_filled_count

        # Rate: how many new slots filled per turn on average
        # Derived from the slope of the slot_curve
        if len(slot_curve) >= 2 and total_slot_ids:
            total = len(total_slot_ids)
            # Convert curve (fractions) back to absolute counts per turn
            abs_curve = [round(v * total) for v in slot_curve]
            # Count turns where at least 1 new slot was filled
            active_turns = sum(
                1 for t in range(1, len(abs_curve))
                if abs_curve[t] > abs_curve[t - 1]
            )
            fill_rate = slots_filled_count / len(slot_curve)  # avg slots/turn
            metrics["slot_fill_rate"] = round(fill_rate, 3)
            metrics["slot_active_turns"] = active_turns
        else:
            metrics["slot_fill_rate"]    = 0.0
            metrics["slot_active_turns"] = 0

        print(f"\n[11] Slot Fill Progress  : {slots_filled_count} slots filled"
              f"  |  avg {metrics['slot_fill_rate']:.2f} slots/turn"
              f"  |  {metrics['slot_active_turns']} turns with new information")

        # Show tier breakdown
        if slot_by_id and filled_slot_ids:
            t1_filled = [slot_by_id[sid]["name"] for sid in filled_slot_ids
                         if sid in (total_slot_ids or set())
                         and slot_by_id.get(sid, {}).get("tier") == 1]
            t2_filled = [slot_by_id[sid]["name"] for sid in filled_slot_ids
                         if sid in (total_slot_ids or set())
                         and slot_by_id.get(sid, {}).get("tier") == 2]
            if t1_filled:
                print(f"    Critical slots filled: {', '.join(t1_filled)}")
            if t2_filled:
                print(f"    Important slots filled: {', '.join(t2_filled)}")
    else:
        metrics["slot_fill_count"]   = None
        metrics["slot_fill_rate"]    = None
        metrics["slot_active_turns"] = None
        
    #  NEW 10. Slot Precision
    # patient-record precision: did recognized doctor questions target slots that
    # are present in this patient's hidden ground-truth record?
    if asked_slot_ids is not None and total_slot_ids:
        slot_prec = _slot_precision(asked_slot_ids, total_slot_ids)
        metrics["slot_precision"] = slot_prec
        print(f"\n[10] Patient-Record Slot Precision: {slot_prec*100:.1f}%  "
              f"(doctor asked about {len(asked_slot_ids & total_slot_ids)}"
              f"/{len(asked_slot_ids)} patient-record slots)")
    else:
        metrics["slot_precision"] = None

    # guide precision: did recognized doctor questions follow the global
    # disease-level guide? This is the fairer precision metric for the
    # independent-doctor setup.
    if asked_slot_ids is not None and guided_slot_ids is not None and asked_slot_ids:
        guided_hits = asked_slot_ids & guided_slot_ids
        guide_prec = len(guided_hits) / len(asked_slot_ids)
        metrics["slot_guide_precision"] = round(guide_prec, 3)
        print(f"[10b] Guide Slot Precision : {guide_prec*100:.1f}%  "
              f"(doctor asked about {len(guided_hits)}"
              f"/{len(asked_slot_ids)} globally guided slots)")
    else:
        metrics["slot_guide_precision"] = None

    print(f"{'='*60}\n")
    return metrics

In [40]:
import csv
import os
from datetime import datetime

# Global log — accumulates rows across all simulations in a run
CONVERSATION_LOG = []


def log_conversation(
    run_label,          # e.g. "Pipeline-Anxious" or "Baseline-Task-Driven"
    disease_name,
    persona_label,
    conversation_history,
    metrics_dict,
    slot_curve=None,
 ):
    """
    Append one simulation's conversation to the global CONVERSATION_LOG.
    Each row = one turn (one speaker message).

    Columns:
        run_label       | pipeline or baseline + persona
        disease_name    | ground truth disease
        persona_label   | Anxious / Task-Driven
        turn_number     | 1, 2, 3 ...
        role            | PATIENT or DOCTOR
        message         | actual text
        word_count      | words in this message
        --- metrics (only on last row of each simulation) ---
        accuracy        | 1 or 0
        final_diagnosis | what model predicted
        symptom_coverage| 0.0 - 1.0
        velocity        | turn number
        slot_recall_t1  | tier-1 slot recall
        slot_precision  | slot precision
        empathy_score   | 0-1 scaled LLM judge score
        slot_curve      | e.g. "0.00→0.08→0.17"
    """
    for turn_idx, msg in enumerate(conversation_history):
        role = "PATIENT" if msg["role"] == "user" else "DOCTOR"
        is_last = (turn_idx == len(conversation_history) - 1)

        row = {
            "run_label":        run_label,
            "disease_name":     disease_name,
            "persona_label":    persona_label,
            "turn_number":      turn_idx + 1,
            "role":             role,
            "message":          msg["content"].replace("\n", " ").strip(),
            "word_count":       len(msg["content"].split()),
            # Metrics — only populate on the final message row
            "accuracy":         metrics_dict.get("accuracy_bool", "") if is_last else "",
            "final_diagnosis":  metrics_dict.get("final_diagnosis", "") if is_last else "",
            "symptom_coverage": metrics_dict.get("symptom_coverage", "") if is_last else "",
            "velocity":         metrics_dict.get("velocity", "") if is_last else "",
            "anxiety_drop":     metrics_dict.get("anxiety_drop", "") if is_last else "",
            "repetition_rate":  metrics_dict.get("repetition_rate", "") if is_last else "",
            "empathy_score":    metrics_dict.get("empathy_score", "") if is_last else "",
            "slot_recall_t1":   metrics_dict.get("slot_recall_tier1", "") if is_last else "",
            "slot_precision":   metrics_dict.get("slot_precision", "") if is_last else "",
            "slot_fill_count":  metrics_dict.get("slot_fill_count", "") if is_last else "",
            "slot_curve":       ("→".join(f"{v:.2f}" for v in slot_curve)
                                 if (is_last and slot_curve) else ""),
        }
        CONVERSATION_LOG.append(row)


def save_conversations_to_csv(output_path=None):
    """
    Write CONVERSATION_LOG to a CSV file.
    Call this after all pipeline_results and baseline_results are collected.

    Args:
        output_path: file path. Defaults to /kaggle/working/conversations_<timestamp>.csv
    """
    if not CONVERSATION_LOG:
        print("No conversations logged yet. Run simulations first.")
        return

    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"/kaggle/working/conversations_{ts}.csv"

    fieldnames = [
        "run_label", "disease_name", "persona_label",
        "turn_number", "role", "message", "word_count",
        "accuracy", "final_diagnosis", "symptom_coverage",
        "velocity", "anxiety_drop", "repetition_rate",
        "empathy_score", "slot_recall_t1", "slot_precision", "slot_fill_count", "slot_curve",
    ]

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(CONVERSATION_LOG)

    print(f"\n Conversations saved to: {output_path}")
    print(f"  Rows: {len(CONVERSATION_LOG)}"
          f"  |  Simulations: {sum(1 for r in CONVERSATION_LOG if r['accuracy'] != '')}")
    return output_path

## Main execution loop

In [41]:
def run_simulation(
    patient_base_prompt,
    initial_message,
    symptom_profile,
    turns=8,
    enable_audio=False,
    jsonl_observations=None,
    slot_record=None,
    total_slot_ids=None,
    priority_slot_ids=None,
    slot_by_id=None,
    keyword_map=None,
 ):
    disease_match  = re.search(r'\*\*(.*?)\*\*', symptom_profile)
    target_disease = disease_match.group(1) if disease_match else "Unknown"
    gt_symptoms    = extract_ground_truth_symptoms(symptom_profile)

    slot_mode = (slot_record is not None and slot_by_id is not None and keyword_map is not None)

    print(f"\n{'─'*60}")
    print(f"  Simulating: {target_disease}  |  {turns} turns"
          f"  |  Slot mode: {'ON' if slot_mode else 'OFF'}")
    print(f"{'─'*60}")

    patient_specific_prompt = (
        patient_base_prompt
        + f"""
    **YOUR HIDDEN MEDICAL TRUTH:**
    {symptom_profile}
    **INSTRUCTION:** Describe your symptoms vividly in response to the doctor's questions.
    If asked about Negative Symptoms listed above, deny them firmly.
    """
    )

    conversation_history = [{"role": "user", "content": initial_message}]
    print(f"\n[PATIENT — Turn 1]\n{initial_message}")
    if enable_audio:
        generate_audio(initial_message, "PATIENT", 1)

    last_hypotheses         = {}
    collected_symptoms_list = []
    state_log               = []

    filled_slot_ids = set()
    asked_slot_ids  = set()
    guided_slot_ids_all = set()
    slot_curve      = []

    #  FIX 4: Build complete total_slot_ids from ALL slots in the record 
    # Previously this was passed in from outside as priority_slot_ids only.
    # We now compute it here from the full slot_record keys + the slot registry.
    if slot_mode and slot_record and slot_by_id:
        name_to_id = {info["name"]: sid for sid, info in slot_by_id.items()}
        all_record_slot_ids = set()
        for obs_name in slot_record:
            if obs_name in name_to_id:
                all_record_slot_ids.add(name_to_id[obs_name])
        # Use ALL record slots as the ground truth denominator
        total_slot_ids = all_record_slot_ids if all_record_slot_ids else total_slot_ids

    for i in range(turns):
        print(f"\n{'·'*40}")
        print(f"  Turn {i+1}")

        #  A: State Manager 
        trigger_prompt = "Analyze the conversation so far and provide the structured JSON report."
        context_parts  = []
        if last_hypotheses:
            context_parts.append(
                f"Last hypotheses: {last_hypotheses}. Update confidence scores."
            )
        if collected_symptoms_list:
            context_parts.append(
                f"Previously gathered symptoms: {collected_symptoms_list}. Add only NEW ones."
            )
        if context_parts:
            trigger_prompt += "\n\n[PREVIOUS CONTEXT]: " + " ".join(context_parts)

        state_messages = (
            [{"role": "system", "content": state_manager_system_prompt}]
            + conversation_history
            + [{"role": "user", "content": trigger_prompt}]
        )
        state_raw  = doctor_pipeline(state_messages, max_new_tokens=400)[0]['generated_text']
        state_data = parse_json_garbage(state_raw)

        current_state_info = "State: Information Gathering"
        max_conf           = 0

        if state_data:
            persona      = state_data.get('patient_persona', 'Unknown')
            hypotheses   = state_data.get('diagnosis_hypotheses', {})
            new_symptoms = state_data.get('gathered_symptoms', [])
            reasoning    = state_data.get('symptom_analysis_reasoning', '')

            seen = {s.lower() for s in collected_symptoms_list}
            for s in new_symptoms:
                if isinstance(s, str) and s.lower() not in seen:
                    collected_symptoms_list.append(s)
                    seen.add(s.lower())

            hypotheses = _safe_hypotheses(hypotheses)
            if not hypotheses and last_hypotheses:
                hypotheses = last_hypotheses
            if hypotheses:
                last_hypotheses = hypotheses

            for _, c in _safe_hypotheses(hypotheses).items():
                try:
                    val = int(str(c).replace('%', ''))
                    if val > max_conf:
                        max_conf = val
                except Exception:
                    continue

            slot_fill_ready = True
            slot_recall_now = 0.0
            tier1_recall_now = 0.0
            slots_filled_now = 0
            min_slots_before_dx = 0
            if slot_mode and total_slot_ids:
                slots_filled_now = len(filled_slot_ids & total_slot_ids)
                slot_recall_now = slots_filled_now / len(total_slot_ids)
                tier1_total = {
                    sid for sid in total_slot_ids
                    if slot_by_id.get(sid, {}).get("tier") == 1
                }
                tier1_recall_now = (
                    len(filled_slot_ids & tier1_total) / len(tier1_total)
                    if tier1_total else 1.0
                )
                min_slots_before_dx = min(4, len(total_slot_ids))
                slot_fill_ready = (
                    (
                        slots_filled_now >= min_slots_before_dx
                        and tier1_recall_now >= 0.60
                    )
                    or i >= turns - 2
                )

            state = (
                "diagnosis_proposal"
                if max_conf > 60 and slot_fill_ready
                else "information_gathering"
            )

            #  A2: Slot mapper — match new symptoms to slot IDs 
            if slot_mode and total_slot_ids:
                for symptom_str in new_symptoms:
                    sid = extract_doctor_intent_slot(symptom_str, keyword_map, slot_by_id)
                    if sid and sid in total_slot_ids:
                        filled_slot_ids.add(sid)

            slot_status = ""
            if slot_mode and total_slot_ids:
                slot_status = (
                    f"\nslot_fill_status: {slots_filled_now}/{len(total_slot_ids)} "
                    f"overall={slot_recall_now:.2f}, tier1={tier1_recall_now:.2f}; "
                    f"diagnosis_allowed={slot_fill_ready}; "
                    f"requires_slots>={min_slots_before_dx}_and_tier1>=0.60_or_late_turn"
                )

            current_state_info = (
                f"Patient Persona: {persona}\n"
                f"conversation_state: {state}\n"
                f"collected_symptoms: {collected_symptoms_list}\n"
                f"Reasoning: {reasoning}\n"
                f"Diagnosis Hypotheses: {hypotheses}"
                f"{slot_status}"
            )
            state_log.append({'turn': i + 1, 'max_conf': max_conf, 'state': state})
            print(f"  [State] {state}  |  Max conf: {max_conf}%  |  Persona: {persona}")
        else:
            print("  [State Manager: JSON parse error — carrying over previous state]")
            state_log.append({'turn': i + 1, 'max_conf': 0, 'state': 'error'})

        #  B: Doctor — inject unfilled slots hint 
        # Once diagnosis is allowed, stop showing remaining slot hints so the
        # doctor actually proposes a diagnosis instead of continuing questions.
        if state == "diagnosis_proposal":
            unfilled_hint = "all key clinical areas covered — propose the most likely diagnosis."
            hint_words_set = set()
            hint_slot_ids_set = set()
        elif slot_mode and total_slot_ids:
            guided_candidate_slot_ids = select_guided_candidate_slots(
                last_hypotheses,
                filled_slot_ids,
                slot_by_id,
                disease_slot_kb=globals().get("DISEASE_SLOT_KB", {}),
                collected_symptoms=collected_symptoms_list,
                keyword_map=keyword_map,
                min_symptom_slots=2,
            )
            unfilled_hint = get_unfilled_slots_hint(
                filled_slot_ids, slot_by_id, slot_record=None,
                candidate_slot_ids=guided_candidate_slot_ids,
                question_bank=globals().get("SLOT_QUESTION_BANK", {})
            )
            hint_words_set = set()
            hint_slot_ids_set = set()
            name_to_id = {info["name"]: sid for sid, info in slot_by_id.items()}
            for line in unfilled_hint.splitlines():
              # Slot name is the text between bullet and arrow
              m = re.search(r'•\s+(.+?)\s+(?:→|->)', line)
              if m:
                  hinted_slot_name = m.group(1).strip()
                  if hinted_slot_name in name_to_id:
                      hint_slot_ids_set.add(name_to_id[hinted_slot_name])
                  for word in hinted_slot_name.lower().split():
                      if len(word) > 3:
                          hint_words_set.add(word)
            guided_slot_ids_all.update(hint_slot_ids_set)

        else:
            unfilled_hint = "use clinical judgment"
            hint_words_set = set()
            hint_slot_ids_set = set()

        # Build a compact forbidden list for the doctor
        forbidden = (", ".join(collected_symptoms_list[:8])
                   if collected_symptoms_list else "none yet")
        
        final_doctor_prompt = (
          doctor_system_prompt
          .replace("{state_details}", current_state_info)
          .replace("{unfilled_slots_hint}", unfilled_hint)
          .replace("{collected_symptoms_explicit}", forbidden)
        )
        doctor_messages = (
            [{"role": "system", "content": final_doctor_prompt}]
            + conversation_history
        )
        doctor_response = doctor_pipeline(doctor_messages)[0]['generated_text'].strip()
        print(f"\n[DOCTOR — Turn {i+1}]\n{doctor_response}")
        if enable_audio:
            generate_audio(doctor_response, "DOCTOR", i + 1)

        #  B2: Intent extractor — which slot did doctor ask about? 
        asked_sid = None
        if slot_mode:
            doctor_question_text = extract_doctor_question_text(doctor_response)
            asked_sid = extract_doctor_intent_slot(
                doctor_question_text, keyword_map, slot_by_id,
                candidate_slot_ids=hint_slot_ids_set if hint_slot_ids_set else None,
            )
            if doctor_question_text:
                print(f"  [Slot intent text] {doctor_question_text}")
            else:
                print("  [Slot intent text] [no explicit question]")
            if asked_sid:
                asked_slot_ids.add(asked_sid)
                if total_slot_ids and asked_sid in total_slot_ids:
                    print(f"  [Slot asked] {slot_by_id[asked_sid]['name']}"
                          f" (tier {slot_by_id[asked_sid]['tier']})")
                else:
                    print(f"  [Slot asked but not in record] {slot_by_id[asked_sid]['name']}"
                          f" (tier {slot_by_id[asked_sid]['tier']})")
            elif doctor_question_text and hint_slot_ids_set:
                hinted_names = [slot_by_id[sid]["name"] for sid in sorted(hint_slot_ids_set)]
                print(f"  [Slot asked] No guided slot matched. Current guided options: {hinted_names}")

        conversation_history.append({"role": "assistant", "content": doctor_response})

        #  C: Patient — slot-aware grounded response 
        if slot_mode and asked_sid:
            patient_response, was_grounded = get_slot_aware_patient_answer(
                asked_sid, slot_record, slot_by_id,
                patient_base_prompt, conversation_history, patient_specific_prompt
            )
            if was_grounded:
                if total_slot_ids and asked_sid in total_slot_ids:
                    filled_slot_ids.add(asked_sid)
                    slot_name = slot_by_id[asked_sid]['name']
                    print(f"  [Slot filled] {slot_name}"
                          f" → {slot_record.get(slot_name, '?')}")
        else:
            patient_view = [
                {"role": "assistant" if m["role"] == "user" else "user",
                 "content": m["content"]}
                for m in conversation_history
            ]
            patient_messages = (
                [{"role": "system", "content": patient_specific_prompt}]
                + patient_view
            )
            patient_response = patient_pipeline(patient_messages)[0]['generated_text'].strip()

        print(f"\n[PATIENT — Turn {i+2}]\n{patient_response}")
        if enable_audio:
            generate_audio(patient_response, "PATIENT", i + 2)
        conversation_history.append({"role": "user", "content": patient_response})

        #  FIX 3: slot_curve appended AFTER patient response 
        # Previously this was before Step B, so the slot filled by the patient
        # in this turn was never counted in the curve point for that same turn.
        if slot_mode and total_slot_ids:
            turn_recall = len(filled_slot_ids & total_slot_ids) / len(total_slot_ids)
            slot_curve.append(round(turn_recall, 3))

        if slot_mode:
            filled_names = [slot_by_id[sid]["name"] for sid in sorted(filled_slot_ids) if sid in slot_by_id]
            asked_names = [slot_by_id[sid]["name"] for sid in sorted(asked_slot_ids) if sid in slot_by_id]
            print(f"\n\n Slots : {filled_slot_ids}")
            print(f" Filled Slot Names : {filled_names}")
            print(f" Asked Slots : {asked_slot_ids}")
            print(f" Asked Slot Names : {asked_names}")
        

    result_metrics = calculate_metrics(
        target_disease,
        gt_symptoms,
        collected_symptoms_list,
        last_hypotheses,
        conversation_history,
        state_log,
        jsonl_observations=jsonl_observations,
        filled_slot_ids=filled_slot_ids  if slot_mode else None,
        asked_slot_ids=asked_slot_ids    if slot_mode else None,
        total_slot_ids=total_slot_ids    if slot_mode else None,
        slot_curve=slot_curve            if slot_mode else None,
        slot_by_id=slot_by_id            if slot_mode else None,
        hint_words=hint_words_set        if slot_mode else None,
        guided_slot_ids=guided_slot_ids_all if slot_mode else None,
    )

    return result_metrics, conversation_history

In [ ]:
def run_batch_evaluation(patient_prompt, scenarios, prompt_label="", turns=8):
    all_metrics = []

    print("\n" + "=" * 65)
    print(f"  BATCH EVALUATION  |  Persona: {prompt_label or 'Unknown'}"
          f"  |  {len(scenarios)} scenarios")
    print("=" * 65)

    for idx, case in enumerate(scenarios):
        print(f"\n>>> Scenario {idx+1}/{len(scenarios)}: {case['name']}")
        m, convo = run_simulation(
            patient_prompt,
            case["opening_message"],
            case["symptom_profile"],
            turns=turns,
            enable_audio=False,
            jsonl_observations=case.get("jsonl_observations"),
            #  NEW — pass slot fields if available (JSONL scenarios only)
            slot_record=case.get("slot_record"),
            total_slot_ids=None,
            slot_by_id=SLOT_BY_ID if case.get("slot_record") else None,
            keyword_map=KEYWORD_MAP if case.get("slot_record") else None,
        )
        m["disease_name"]  = case["name"]
        m["persona_label"] = prompt_label
        m["source"]        = case.get("source", "handcrafted")
        all_metrics.append(m)
        log_conversation(
            run_label=f"Pipeline-{prompt_label}",
            disease_name=case["name"],
            persona_label=prompt_label,
            conversation_history=convo,
            metrics_dict=m,
            slot_curve=m.get("slot_curve"),
        )

    # Aggregate
    n         = len(all_metrics)
    total_acc = sum(m['accuracy_bool'] for m in all_metrics)
    avg_cov   = sum(m['symptom_coverage'] for m in all_metrics) / n
    avg_words = sum(m['avg_word_count'] for m in all_metrics) / n
    avg_vel   = sum(m['velocity'] for m in all_metrics) / n
    avg_drop  = sum(m['anxiety_drop'] for m in all_metrics) / n
    avg_rep   = sum(m['repetition_rate'] for m in all_metrics) / n
    avg_red   = sum(m['question_redundancy'] for m in all_metrics) / n

    empathy_scores = [m['empathy_score'] for m in all_metrics if m.get('empathy_score') is not None]
    avg_empathy = sum(empathy_scores) / len(empathy_scores) if empathy_scores else None

    f1_scores = [m['extraction_f1'] for m in all_metrics if m.get('extraction_f1') is not None]
    avg_f1    = sum(f1_scores) / len(f1_scores) if f1_scores else None

    #  NEW — Slot metric aggregation
    slot_recall_scores = [m['slot_recall_overall'] for m in all_metrics
                          if m.get('slot_recall_overall') is not None]
    slot_t1_scores     = [m['slot_recall_tier1'] for m in all_metrics
                          if m.get('slot_recall_tier1') is not None]
    slot_prec_scores   = [m['slot_precision'] for m in all_metrics
                          if m.get('slot_precision') is not None]

    avg_slot_recall = sum(slot_recall_scores)/len(slot_recall_scores) if slot_recall_scores else None
    avg_slot_t1     = sum(slot_t1_scores)/len(slot_t1_scores)         if slot_t1_scores     else None
    avg_slot_prec   = sum(slot_prec_scores)/len(slot_prec_scores)     if slot_prec_scores   else None

    print("\n" + "=" * 65)
    print("  AGGREGATED RESULTS")
    print("=" * 65)
    print(f"  Diagnostic Accuracy     : {total_acc}/{n}  ({total_acc/n*100:.1f}%)")
    print(f"  Symptom Coverage        : {avg_cov*100:.1f}%")
    print(f"  Doctor Word Count       : {avg_words:.1f} words/turn")
    print(f"  Diagnostic Velocity     : {avg_vel:.1f} turns")
    print(f"  Anxiety Drop (avg)      : {avg_drop:.2f}")
    print(f"  Repetition Rate (avg)   : {avg_rep:.4f}")
    print(f"  Question Redundancy     : {avg_red*100:.1f}%")
    if avg_empathy is not None:
        print(f"  Empathy Score (avg)     : {avg_empathy:.3f}")
    if avg_f1 is not None:
        print(f"  Extraction F1 (JSONL)   : {avg_f1:.3f}")
    # NEW
    if avg_slot_recall is not None:
        print(f"  Slot Recall (overall)   : {avg_slot_recall*100:.1f}%")
        print(f"  Slot Recall (tier-1)    : {avg_slot_t1*100:.1f}%")
        print(f"  Slot Precision          : {avg_slot_prec*100:.1f}%")

    # Per-scenario table
    col = "{:<28} | {:<5} | {:<5} | {:<5} | {:<4} | {:<7} | {:<7} | {:<7}"
    print("\n" + col.format("Disease","Acc","Cov%","Words","Turn","SlotRec","SlotT1","SlotPrc"))
    print("─" * 90)
    for m in all_metrics:
        sr   = f"{m['slot_recall_overall']*100:.0f}%" if m.get('slot_recall_overall') is not None else "N/A"
        st1  = f"{m['slot_recall_tier1']*100:.0f}%"  if m.get('slot_recall_tier1')   is not None else "N/A"
        sprc = f"{m['slot_precision']*100:.0f}%"     if m.get('slot_precision')       is not None else "N/A"
        print(col.format(
            m["disease_name"][:27],
            "Match" if m["accuracy_bool"] else "Miss",
            f"{m['symptom_coverage']*100:.0f}",
            f"{m['avg_word_count']:.0f}",
            str(m['velocity']),
            sr, st1, sprc,
        ))

    return all_metrics

In [43]:
def run_baseline_simulation(
    patient_base_prompt, initial_message, symptom_profile,
    turns=8, jsonl_observations=None,
):
    """Baseline: plain LLaMA doctor with no system prompt and no state manager."""
    disease_match  = re.search(r'\*\*(.*?)\*\*', symptom_profile)
    target_disease = disease_match.group(1) if disease_match else "Unknown"
    gt_symptoms    = extract_ground_truth_symptoms(symptom_profile)
 
    patient_specific_prompt = (
        patient_base_prompt
        + f"""
    **YOUR HIDDEN MEDICAL TRUTH:**
    {symptom_profile}
    **INSTRUCTION:** Describe your symptoms vividly. Deny Negative Symptoms when asked.
    """
    )
 
    conversation_history    = [{"role": "user", "content": initial_message}]
    last_hypotheses         = {}
    collected_symptoms_list = []
    state_log               = []
 
    for i in range(turns):
        trigger_prompt = "Analyze the conversation so far and provide the structured JSON report."
        if last_hypotheses:
            trigger_prompt += f" Last hypotheses: {last_hypotheses}."
        if collected_symptoms_list:
            trigger_prompt += f" Previous symptoms: {collected_symptoms_list}."
 
        state_messages = (
            [{"role": "system", "content": state_manager_system_prompt}]
            + conversation_history
            + [{"role": "user", "content": trigger_prompt}]
        )
        state_raw  = doctor_pipeline(state_messages, max_new_tokens=400)[0]['generated_text']
        state_data = parse_json_garbage(state_raw)
 
        max_conf = 0
        if state_data:
            hypotheses   = _safe_hypotheses(state_data.get('diagnosis_hypotheses', {}))
            new_symptoms = state_data.get('gathered_symptoms', [])
            seen = {s.lower() for s in collected_symptoms_list}
            for s in new_symptoms:
                if isinstance(s, str) and s.lower() not in seen:
                    collected_symptoms_list.append(s)
                    seen.add(s.lower())
            if not hypotheses and last_hypotheses:
                hypotheses = last_hypotheses
            if hypotheses:
                last_hypotheses = hypotheses
            for _, c in _safe_hypotheses(hypotheses).items():
                try:
                    val = int(str(c).replace('%',''))
                    if val > max_conf:
                        max_conf = val
                except Exception:
                    continue
            state = "diagnosis_proposal" if max_conf > 60 else "information_gathering"
            state_log.append({'turn': i+1, 'max_conf': max_conf, 'state': state})
        else:
            state_log.append({'turn': i+1, 'max_conf': 0, 'state': 'error'})
 
        # Baseline doctor — no system prompt
        doctor_response = doctor_pipeline(
            conversation_history, max_new_tokens=150,
        )[0]['generated_text'].strip()
        conversation_history.append({"role": "assistant", "content": doctor_response})
 
        patient_view = [
            {"role": "assistant" if m["role"] == "user" else "user", "content": m["content"]}
            for m in conversation_history
        ]
        patient_messages = (
            [{"role": "system", "content": patient_specific_prompt}] + patient_view
        )
        patient_response = patient_pipeline(patient_messages)[0]['generated_text'].strip()
        conversation_history.append({"role": "user", "content": patient_response})
 
    result_metrics =  calculate_metrics(
        target_disease, gt_symptoms, collected_symptoms_list,
        last_hypotheses, conversation_history, state_log,
        jsonl_observations=jsonl_observations,
    )

    return result_metrics, conversation_history


In [44]:
def run_baseline_batch(patient_prompt, scenarios, prompt_label="", turns=8):
    all_metrics = []
    print(f"\n{'='*65}")
    print(f"  BASELINE BATCH  |  Persona: {prompt_label}  |  {len(scenarios)} scenarios")
    print(f"{'='*65}")
 
    for idx, case in enumerate(scenarios):
        print(f"\n>>> [Baseline] Scenario {idx+1}/{len(scenarios)}: {case['name']}")
        m, convo = run_baseline_simulation(     # ← unpack tuple
            patient_base_prompt=patient_prompt,
            initial_message=case["opening_message"],
            symptom_profile=case["symptom_profile"],
            turns=turns,
            jsonl_observations=case.get("jsonl_observations"),
        )
        m["disease_name"]  = case["name"]
        m["persona_label"] = prompt_label
        m["source"]        = case.get("source", "handcrafted")
        all_metrics.append(m)
 
        # ── FIX 2: Log baseline conversations to CSV ──────────────────────
        log_conversation(
            run_label=f"Baseline-{prompt_label}",
            disease_name=case["name"],
            persona_label=prompt_label,
            conversation_history=convo,
            metrics_dict=m,
            slot_curve=None,   # baseline has no slot tracking
        )
 
    return all_metrics

In [45]:
METRIC_DEFS = [
    ("accuracy_bool",        "Diagnostic Accuracy",      True),
    ("symptom_coverage",     "Symptom Coverage",         True),
    ("avg_word_count",       "Doctor Word Count",        False),
    ("velocity",             "Diagnostic Velocity",      False),
    ("anxiety_drop",         "Anxiety Drop",             True),
    ("repetition_rate",      "Repetition Rate",          False),
    ("empathy_score",        "Empathy Score",            True),
    ("question_redundancy",  "Question Redundancy",      False),
    ("extraction_f1",        "Extraction F1 (JSONL)",    True),
    ("slot_recall_overall",  "Slot Recall (overall)",    True),   
    ("slot_recall_tier1",    "Slot Recall (tier-1)",     True),   
    ("slot_precision",       "Patient Slot Precision",   True),   
    ("slot_guide_precision", "Guide Slot Precision",     True),
    ("slot_fill_count",  "Slot Fill Count",     True),
    ("slot_fill_rate",   "Slot Fill Rate",      True),
]
 

In [46]:
def _avg(records, key):
    vals = [r[key] for r in records if r.get(key) is not None]
    return sum(vals) / len(vals) if vals else None
 

In [47]:
def print_overall_summary(all_results, label="OUR PIPELINE"):
    persona_names = list(all_results.keys())
    persona_avgs  = {p: {k: _avg(r, k) for k, _, _ in METRIC_DEFS}
                     for p, r in all_results.items()}
    all_flat      = [r for recs in all_results.values() for r in recs]
    overall_avgs  = {k: _avg(all_flat, k) for k, _, _ in METRIC_DEFS}

    col_w, label_w = 14, 30
    header = f"  {'Metric':<{label_w}}"
    for p in persona_names:
        header += f"  {p[:col_w]:>{col_w}}"
    header += f"  {'OVERALL':>{col_w}}"
    sep = "─" * len(header)

    print(f"\n\n{'='*len(header)}")
    print(f"  {label}  —  {len(all_flat)} simulations")
    print(f"{'='*len(header)}")
    print(header)
    print(sep)

    for key, lbl, _ in METRIC_DEFS:
        row = f"  {lbl:<{label_w}}"
        for p in persona_names:
            v = persona_avgs[p][key]
            row += f"  {'N/A':>{col_w}}" if v is None else f"  {v:>{col_w}.3f}"
        ov = overall_avgs[key]
        row += f"  {'N/A':>{col_w}}" if ov is None else f"  {ov:>{col_w}.3f}"
        print(row)
    print(sep)

    THRESHOLDS = [
        ("accuracy_bool",       "≥ 0.55  Diagnostic Accuracy",   lambda v: v >= 0.55),  # was 0.70
        ("symptom_coverage",    "≥ 0.50  Symptom Coverage",      lambda v: v >= 0.50),  # was 0.60
        ("avg_word_count",      "≤ 45    Words/turn",            lambda v: v <= 45),    # unchanged
        ("anxiety_drop",        "≥ 0.10  Anxiety Drop",          lambda v: v >= 0.10),  # was 1.0 (unrealistic)
        ("repetition_rate",     "≤ 0.15  Repetition Rate",       lambda v: v <= 0.15),  # was 0.05 (too strict)
        ("empathy_score",       "≥ 0.50  Empathy Score",         lambda v: v >= 0.50),
        ("question_redundancy", "≤ 0.60  Question Redundancy",   lambda v: v <= 0.60),  # was 0.30
        ("slot_recall_tier1",   "≥ 0.20  Slot Recall Tier-1",    lambda v: v >= 0.20),  # was 0.50
        ("slot_precision",      "≥ 0.25  Slot Precision",        lambda v: v >= 0.25),  # was 0.60
    ]
    print(f"\n  Quality gates:")
    passes = 0
    for key, thr_label, test_fn in THRESHOLDS:
        v = overall_avgs.get(key)
        if v is None:
            continue
        ok = test_fn(v)
        passes += ok
        print(f"    {'PASS' if ok else 'FAIL'}  {thr_label:<38}  (actual: {v:.3f})")
    print(f"\n  Gates passed: {passes}/{len(THRESHOLDS)}")
    print()
    return overall_avgs

In [48]:
def print_calibrated_comparison(pipeline_results, baseline_results):
    p_flat = [r for recs in pipeline_results.values() for r in recs]
    b_flat = [r for recs in baseline_results.values() for r in recs]
    p_avgs = {k: _avg(p_flat, k) for k, _, _ in METRIC_DEFS}
    b_avgs = {k: _avg(b_flat, k) for k, _, _ in METRIC_DEFS}

    COMPARABLE = [
        ("accuracy_bool",       "Diagnostic Accuracy",   True),
        ("symptom_coverage",    "Symptom Coverage",      True),
        ("avg_word_count",      "Doctor Word Count",     False),
        ("velocity",            "Diagnostic Velocity",   False),
        ("anxiety_drop",        "Anxiety Drop",          True),
        ("repetition_rate",     "Repetition Rate",       False),
        ("empathy_score",       "Empathy Score",         True),
        ("question_redundancy", "Question Redundancy",   False),
        ("extraction_f1",       "Extraction F1 (JSONL)", True),
    ]

    col_w, label_w = 12, 30
    header = (f"  {'Metric':<{label_w}}  {'Baseline':>{col_w}}  "
              f"{'Pipeline':>{col_w}}  {'Δ':>{col_w}}  {'Result':>10}")
    sep = "─" * len(header)

    print(f"\n\n{'='*len(header)}")
    print(f"  CALIBRATED COMPARISON:  Baseline (plain LLaMA)  vs  Our Pipeline")
    print(f"  Scenarios: {len(p_flat)} pipeline sims  |  {len(b_flat)} baseline sims")
    print(f"{'='*len(header)}")
    print(header)
    print(sep)

    gain_scores = []
    for key, lbl, higher_is_better in COMPARABLE:
        p_val = p_avgs.get(key)
        b_val = b_avgs.get(key)
        if p_val is None or b_val is None:
            print(f"  {lbl:<{label_w}}  {'N/A':>{col_w}}  {'N/A':>{col_w}}"
                  f"  {'N/A':>{col_w}}  {'—':>10}")
            continue
        delta = p_val - b_val
        gain = delta if higher_is_better else -delta
        gain_scores.append(gain)
        rel = (gain / abs(b_val) * 100) if b_val != 0 else 0.0
        result_str = ("  TIE" if abs(gain) < 0.001 else
                      (f"  ↑ +{rel:.1f}%" if gain > 0 else f"  ↓ {rel:.1f}%"))
        print(f"  {lbl:<{label_w}}  {b_val:>{col_w}.3f}  {p_val:>{col_w}.3f}"
              f"  {delta:>+{col_w}.3f}  {result_str:>10}")

    print(sep)
    if gain_scores:
        composite = sum(gain_scores) / len(gain_scores)
        improved = sum(1 for g in gain_scores if g > 0.001)
        degraded = sum(1 for g in gain_scores if g < -0.001)
        tied = len(gain_scores) - improved - degraded
        print(f"\n  Composite improvement score : {composite:+.4f}")
        print(f"  Metrics improved            : {improved}/{len(gain_scores)}")
        print(f"  Metrics degraded            : {degraded}/{len(gain_scores)}")
        print(f"  Metrics tied                : {tied}/{len(gain_scores)}")
        verdict = (
            f"Pipeline outperforms baseline on {improved}/{len(gain_scores)} metrics."
            if improved > degraded else
            f"Baseline wins on {degraded}/{len(gain_scores)} metrics. Review prompts."
            if degraded > improved else
            "Pipeline and baseline perform comparably."
        )
        print(f"\n  VERDICT: {verdict}")

    # ── Pipeline-only slot quality section ───────────────────────────────────
    # Baseline has no slot_record so there is no valid baseline to compare.
    # We show absolute pipeline scores vs fixed quality targets instead.
    jsonl_pipeline = [r for r in p_flat if r.get("slot_recall_overall") is not None]
    if jsonl_pipeline:
        n_slot = len(jsonl_pipeline)
        print(f"\n  {'─'*60}")
        print(f"  SLOT-FILLING QUALITY  (pipeline, {n_slot} JSONL scenarios)")
        print(f"  Baseline N/A — it has no structured slot records to track against.")
        print(f"  {'─'*60}")
        SLOT_SECTION = [
            ("slot_recall_overall", "Slot Recall (overall)",  0.15),  # was 0.30 — too high for 193 slots
            ("slot_recall_tier1",   "Slot Recall (tier-1)",   0.35),  # was 0.50 — tier-1 is 15/193 slots
            ("slot_precision",      "Slot Precision",         0.40),  # was 0.60 — doctor asks broadly
            ("slot_fill_count",     "Slot Fill Count (avg)",   2.0),   # ≥2 slots/sim
            ("slot_fill_rate",      "Slot Fill Rate (slots/turn)", 0.25),  # ≥1 slot per 4 turns
        ]
        col2 = f"  {{:<{label_w}}}  {{:>{col_w}}}  {{:>{col_w}}}  {{:>10}}"
        print(col2.format("Metric", "Pipeline", "Target", "Status"))
        print(f"  {'─'*(label_w + col_w*2 + 16)}")
        for key, lbl, target in SLOT_SECTION:
            v = _avg(jsonl_pipeline, key)
            if v is None:
                print(col2.format(lbl, "N/A", f"≥{target:.2f}", "—"))
            else:
                status = "✓ PASS" if v >= target else "✗ FAIL"
                print(col2.format(lbl, f"{v:.3f}", f"≥{target:.2f}", status))

        # Average info flow curve
        curves = [r["slot_curve"] for r in jsonl_pipeline
                  if r.get("slot_curve") and r["slot_curve"]]
        if curves:
            min_len = min(len(c) for c in curves)
            avg_curve = [round(sum(c[t] for c in curves) / len(curves), 3)
                         for t in range(min_len)]
            print(f"\n  Avg info flow curve : {' → '.join(f'{v:.2f}' for v in avg_curve)}")
            print(f"  (turn-by-turn slot recall — steeper rise = better questioning)")
    else:
        print(f"\n  Slot metrics: no JSONL scenarios with slot records found.")
    print()


def print_calibrated_comparison_per_persona(pipeline_results, baseline_results):
    for persona_name, pipeline_metrics in pipeline_results.items():
        baseline_metrics = baseline_results.get(persona_name, [])
        print(f"\n\n{'='*70}")
        print(f"  CALIBRATED COMPARISON — PERSONA: {persona_name}")
        print(f"{'='*70}")
        print_calibrated_comparison(
            {persona_name: pipeline_metrics},
            {persona_name: baseline_metrics},
        )

In [ ]:
def compare_baseline_vs_finetuned(
    baseline_metrics,
    finetuned_metrics,
    label_a="Baseline",
    label_b="Fine-tuned",
):
    """
    Print a side-by-side comparison of two metric lists
    (e.g. base LLaMA vs Meditron fine-tuned).

    Usage:
        base_m = run_batch_evaluation(anxious_patient_prompt, all_scenarios, "Anxious")
        # reload doctor_pipeline with fine-tuned checkpoint, then:
        ft_m   = run_batch_evaluation(anxious_patient_prompt, all_scenarios, "Anxious")
        compare_baseline_vs_finetuned(base_m, ft_m)
    """
    def avg(metrics, key):
        vals = [m[key] for m in metrics if m.get(key) is not None]
        return sum(vals) / len(vals) if vals else None

    # ── FIX 5: all 11 metrics including the 3 new slot metrics ───────────────
    keys = [
        ("accuracy_bool",       "Diagnostic Accuracy",      True),
        ("symptom_coverage",    "Symptom Coverage",         True),
        ("avg_word_count",      "Doctor Word Count",        False),
        ("velocity",            "Diagnostic Velocity",      False),
        ("anxiety_drop",        "Anxiety Drop",             True),
        ("repetition_rate",     "Repetition Rate",          False),
        ("empathy_score",       "Empathy Score",            True),
        ("question_redundancy", "Question Redundancy",      False),
        ("extraction_f1",       "Extraction F1 (JSONL)",    True),
        ("slot_recall_overall", "Slot Recall (overall)",    True),   
        ("slot_recall_tier1",   "Slot Recall (tier-1)",     True),   
        ("slot_precision",      "Slot Precision",           True),   
    ]

    print("\n" + "=" * 70)
    print(f"  COMPARISON: {label_a}  vs  {label_b}")
    print("=" * 70)
    print(f"  {'Metric':<30}  {label_a:>12}  {label_b:>12}  {'Δ':>8}  {'Better':>10}")
    print("─" * 70)

    for key, label, higher_is_better in keys:
        a_val = avg(baseline_metrics, key)
        b_val = avg(finetuned_metrics, key)

        if a_val is None or b_val is None:
            print(f"  {label:<30}  {'N/A':>12}  {'N/A':>12}  {'N/A':>8}  {'—':>10}")
            continue

        delta = b_val - a_val
        if higher_is_better:
            better = label_b if delta > 0.001 else (label_a if delta < -0.001 else "Tie")
        else:
            better = label_b if delta < -0.001 else (label_a if delta > 0.001 else "Tie")

        print(f"  {label:<30}  {a_val:>12.3f}  {b_val:>12.3f}  {delta:>+8.3f}  {better:>10}")

    print("─" * 70)

In [50]:
def evaluate_extraction_from_jsonl(jsonl_path, max_records=20):
    """
    Evaluate the state manager's ability to extract clinical observations
    directly from clinician transcripts in train.jsonl — no patient simulation
    needed. This is the cleanest way to benchmark NLU quality.
 
    Prints per-record and aggregate Precision / Recall / F1.
    """
    results = []
 
    with open(jsonl_path) as f:
        records = [json.loads(line) for line in f][:max_records]
 
    print(f"\nRunning extraction evaluation on {len(records)} JSONL records...")
 
    for record in records:
        transcript = record.get("transcript", "")
        raw_obs    = record.get("observations", "[]")
        gt_obs     = json.loads(raw_obs) if isinstance(raw_obs, str) else raw_obs
 
        if not transcript or not gt_obs:
            continue
 
        # Ask state manager to extract symptoms from the transcript
        prompt = (
            f"The following is a clinical note. "
            f"Extract all symptoms, findings, and observations as a gathered_symptoms list.\n\n"
            f"Transcript:\n{transcript}"
        )
        state_messages = [
            {"role": "system", "content": state_manager_system_prompt},
            {"role": "user",   "content": prompt},
        ]
        raw    = doctor_pipeline(state_messages)[0]['generated_text']
        parsed = parse_json_garbage(raw)
 
        extracted = parsed.get('gathered_symptoms', []) if parsed else []
        prec, rec, f1 = _observation_extraction_f1(extracted, gt_obs)
 
        results.append({
            "id": record.get("id"),
            "precision": prec,
            "recall": rec,
            "f1": f1,
        })
        print(f"  Record {record.get('id'):>4} | P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}")
 
    if results:
        avg_p  = sum(r['precision'] for r in results) / len(results)
        avg_r  = sum(r['recall']    for r in results) / len(results)
        avg_f1 = sum(r['f1']        for r in results) / len(results)
        print(f"\n  AGGREGATE over {len(results)} records:")
        print(f"  Precision: {avg_p:.3f}  |  Recall: {avg_r:.3f}  |  F1: {avg_f1:.3f}")
 
    return results

In [51]:
JSONL_CONVERSION_PROMPT = """You are a medical scenario formatter. You will be given a clinical nursing note and a list of structured observations about a patient.
 
Your job is to convert this into a patient simulation scenario with THREE fields:
 
1. "name": The disease/condition name (1-5 words, title case). Infer from the observations and transcript. Use the Secondary diagnosis if present.
 
2. "opening_message": A single natural sentence spoken BY THE PATIENT to a doctor. It must:
   - Be in first person ("I have...", "Doctor, I'm feeling...")
   - Describe the most prominent symptom from the transcript
   - Sound like a real patient, not a clinical note
   - Be under 20 words
 
3. "symptom_profile": A formatted block following EXACTLY this template:
   You are suffering from **<disease name>**.
   **Symptoms to Simulate:**
   - <symptom 1>
   - <symptom 2>
   - <symptom 3 (3-6 symptoms total)>
   **Negative Symptoms (Deny these):** <list 2-3 symptoms NOT present, to help differential diagnosis>
 
Output ONLY valid JSON with these exact three keys. No explanation, no markdown fences.
Example output format:
{"name":"Dengue Fever","opening_message":"Doctor, I've had a sudden high fever since yesterday and my whole body is aching.","symptom_profile":"You are suffering from **Dengue Fever**.\\n**Symptoms to Simulate:**\\n- Sudden high fever (104F)\\n- Severe joint and muscle pain\\n- Pain behind the eyes\\n- Skin rash\\n**Negative Symptoms (Deny these):** No cough, no sore throat, no runny nose"}
"""

In [52]:
def convert_jsonl_to_scenarios_via_llm(jsonl_path, max_records=20, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        with open(cache_path) as f:
            cached = json.load(f)
        print(f"Loaded {len(cached)} converted scenarios from cache: {cache_path}")
        return cached
 
    scenarios, failed = [], 0
 
    with open(jsonl_path) as f:
        records = [json.loads(line) for line in f][:max_records]
 
    print(f"\nConverting {len(records)} JSONL records via LLM...")
 
    for idx, record in enumerate(records):
        transcript   = record.get("transcript", "").strip()
        raw_obs      = record.get("observations", "[]")
        observations = json.loads(raw_obs) if isinstance(raw_obs, str) else raw_obs
 
        if not transcript:
            failed += 1
            continue
 
        obs_lines = []
        for obs in observations[:15]:
            val = obs.get("value","")
            val_str = ", ".join(val) if isinstance(val, list) else str(val)
            obs_lines.append(f"  - {obs['name']}: {val_str}")
 
        user_content = (
            f"CLINICAL TRANSCRIPT:\n{transcript[:800]}\n\n"
            f"STRUCTURED OBSERVATIONS:\n{chr(10).join(obs_lines)}\n\n"
            f"Convert the above into the scenario JSON format as instructed."
        )
        messages = [
            {"role": "system", "content": JSONL_CONVERSION_PROMPT},
            {"role": "user",   "content": user_content},
        ]
 
        try:
            raw    = doctor_pipeline(messages, max_new_tokens=300)[0]['generated_text']
            parsed = parse_json_garbage(raw)
 
            if not parsed or not all(k in parsed for k in ("name","opening_message","symptom_profile")):
                print(f"  [{idx+1:>3}/{len(records)}] FAILED — skipping record {record['id']}")
                failed += 1
                continue
 
            scenario = {
                "name":              parsed["name"],
                "opening_message":   parsed["opening_message"],
                "symptom_profile":   parsed["symptom_profile"],
                "jsonl_observations": observations,
                "source":            "jsonl_llm_converted",
            }
            scenarios.append(scenario)
            print(f"  [{idx+1:>3}/{len(records)}] OK  →  {parsed['name']}")
 
        except Exception as e:
            print(f"  [{idx+1:>3}/{len(records)}] ERROR: {e}")
            failed += 1
 
    print(f"\nConversion complete: {len(scenarios)} succeeded, {failed} failed.")
    if cache_path:
        with open(cache_path, 'w') as f:
            json.dump(scenarios, f, indent=2)
        print(f"Cached to: {cache_path}")
    return scenarios
 

In [53]:
JSONL_PATH        = "/kaggle/input/datasets/anyagupta/synurdataset/train.jsonl"
SCHEMA_PATH       = "/kaggle/input/datasets/anyagupta/synur-schema/synur_schema.json"
SCENARIO_CACHE    = "/kaggle/working/converted_scenarios.json"
SLOT_QUESTION_CACHE = "/kaggle/working/slot_question_bank.json"
DISEASE_SLOT_KB_CACHE = "/kaggle/working/disease_slot_kb.json"
DISEASE_SLOT_KB_MAX_RECORDS = None
DISEASE_SLOT_TOP_K = 24
QUESTION_BANK_TIERS = (1, 2, 3)  # generate for tier-1/2; use (1,2,3) for full schema
QUESTION_BANK_LIMIT = None    # set small number for quick tests, None for all selected tiers
MAX_JSONL_CONVERT = 20    # raise to 50+ for full run
HANDCRAFTED_CAP   = 0
TURNS_PER_SIM     = 6     # raise to 5 for full run
 
print("=" * 65)
print("  DOCTOR BOT PIPELINE — WITH SLOT-FILLING STRATEGY")
print(f"  JSONL: {MAX_JSONL_CONVERT}  |  Handcrafted: {HANDCRAFTED_CAP}"
      f"  |  Turns: {TURNS_PER_SIM}")
print("=" * 65)
 


  DOCTOR BOT PIPELINE — WITH SLOT-FILLING STRATEGY
  JSONL: 20  |  Handcrafted: 0  |  Turns: 6


In [54]:
#  NEW — Phase 0: Build slot registry from synur_schema.json
print("\n[PHASE 0] Building slot registry from synur_schema.json...")
SLOT_BY_ID, KEYWORD_MAP, SLOT_NAMES = build_slot_registry(SCHEMA_PATH)

print("\n[PHASE 0b] Building/loading slot question bank...")
SLOT_QUESTION_BANK = build_slot_question_bank(
    SLOT_BY_ID,
    cache_path=SLOT_QUESTION_CACHE,
    tiers=QUESTION_BANK_TIERS,
    limit=QUESTION_BANK_LIMIT,
)
SLOT_BY_ID, KEYWORD_MAP = augment_slot_registry_with_question_bank(
    SLOT_BY_ID, KEYWORD_MAP, SLOT_QUESTION_BANK
)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PHASE 0] Building slot registry from synur_schema.json...
Slot registry built: 193 slots, 292 keywords

[PHASE 0b] Building/loading slot question bank...


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  generated question: Broset violence checklist - confusion -> Have you been feeling confused or disoriented recently, making it hard to think or make decisions?


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  generated question: Oxygen flow rate unit -> What unit is used to measure the flow rate of oxygen you're using?


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  generated question: Bladder scan volume unit -> What unit of measurement is used to report the volume of fluid in your bladder?


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  generated question: Caloric intake unit -> What is your usual way of measuring the amount of calories you consume each day?
  generated question: Emesis volume unit -> How many milliliters of vomit did you usually spit up when you were sick?
Saved 193 slot questions to cache: /kaggle/working/slot_question_bank.json
Augmented slot registry with 886 question-bank keywords


In [55]:
#  NEW — Phase 0c: Build disease-level slot knowledge without patient-specific leakage
print("\n[PHASE 0c] Building/loading disease-slot knowledge base...")
DISEASE_SLOT_KB = build_disease_slot_knowledge(
    JSONL_PATH, SLOT_BY_ID,
    cache_path=DISEASE_SLOT_KB_CACHE,
    max_records=DISEASE_SLOT_KB_MAX_RECORDS,
    top_k=DISEASE_SLOT_TOP_K,
)

#  NEW — Phase 0: Load JSONL scenarios with structured slot records
print("\n[STEP 1a] Loading JSONL scenarios with slot records...")
jsonl_scenarios_with_slots = load_jsonl_scenarios_with_slots(
    JSONL_PATH, SLOT_BY_ID, max_scenarios=MAX_JSONL_CONVERT
)


[PHASE 0c] Building/loading disease-slot knowledge base...
Saved disease-slot knowledge for 60 diseases: /kaggle/working/disease_slot_kb.json
Built disease-slot knowledge for 60 diseases from 100 records

[STEP 1a] Loading JSONL scenarios with slot records...
Loaded 20 scenarios (15 with resolved diagnosis, 5 still 'Clinical Case N')


In [56]:
# Also load LLM-converted scenarios for handcrafted-style eval
print("\n[STEP 1b] LLM-converting JSONL records to scenario format...")
jsonl_scenarios_llm = convert_jsonl_to_scenarios_via_llm(
    JSONL_PATH, max_records=MAX_JSONL_CONVERT, cache_path=SCENARIO_CACHE,
)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[STEP 1b] LLM-converting JSONL records to scenario format...

Converting 20 JSONL records via LLM...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [  1/20] OK  →  Acute Respiratory Failure


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [ 17/20] OK  →  Pneumonia


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [ 18/20] OK  →  Urinary Tract Infection with Complications


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [ 19/20] OK  →  Chronic Heart Failure
  [ 20/20] OK  →  Biliary Obstruction

Conversion complete: 20 succeeded, 0 failed.
Cached to: /kaggle/working/converted_scenarios.json


In [57]:
#  NEW — Merge slot_record fields into LLM-converted scenarios
# So the LLM-converted scenarios also benefit from slot grounding
slot_record_by_idx = {i: s.get("slot_record", {})
                      for i, s in enumerate(jsonl_scenarios_with_slots)}
for i, scenario in enumerate(jsonl_scenarios_llm):
    if i < len(jsonl_scenarios_with_slots):
        src = jsonl_scenarios_with_slots[i]
        scenario["slot_record"]       = src.get("slot_record")
        scenario["total_slots"]       = src.get("total_slots")
        scenario["priority_slot_ids"] = src.get("priority_slot_ids", [])
        scenario["jsonl_observations"] = src.get("jsonl_observations")
 
# Combined pool
all_scenarios = handcrafted_scenarios[:HANDCRAFTED_CAP] + jsonl_scenarios_llm
print(f"\nTotal scenarios: {len(all_scenarios)}"
      f"  ({HANDCRAFTED_CAP} handcrafted + {len(jsonl_scenarios_llm)} LLM-converted"
      f", {sum(1 for s in jsonl_scenarios_llm if s.get('slot_record'))} with slot grounding)")
 



Total scenarios: 20  (0 handcrafted + 20 LLM-converted, 20 with slot grounding)


In [58]:
from collections import defaultdict
# ── Smoke test ────────────────────────────────────────────────────────────────
print("\n[STEP 2] Smoke test — 1 scenario, full pipeline + slot mode...")
test_case = all_scenarios[HANDCRAFTED_CAP]  # first JSONL case
run_simulation(
    patient_base_prompt=anxious_patient_prompt,
    initial_message=test_case["opening_message"],
    symptom_profile=test_case["symptom_profile"],
    turns=TURNS_PER_SIM,
    enable_audio=False,
    jsonl_observations=test_case.get("jsonl_observations"),
    slot_record=test_case.get("slot_record"),
    total_slot_ids=set(test_case.get("priority_slot_ids", [])) or None,
    slot_by_id=SLOT_BY_ID,
    keyword_map=KEYWORD_MAP,
)
 


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[STEP 2] Smoke test — 1 scenario, full pipeline + slot mode...

────────────────────────────────────────────────────────────
  Simulating: Acute Respiratory Failure  |  6 turns  |  Slot mode: ON
────────────────────────────────────────────────────────────

[PATIENT — Turn 1]
Doctor, I've been having trouble breathing and my legs feel numb and tingly since I got sick.

········································
  Turn 1


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [State] information_gathering  |  Max conf: 60%  |  Persona: Emotional/Anxious


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[DOCTOR — Turn 6]
I understand your concerns, and I want to reassure you that we'll do everything we can to confirm the diagnosis and develop a plan to manage your symptoms.
  [Slot intent text] [no explicit question]


Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PATIENT — Turn 7]
Doctor, I appreciate your reassurance, but I'm still scared. What if the diagnosis is correct? What if I really do have heart failure? I've heard of people who have to live with heart failure, and it's not a good life. I'm worried about the future, Doctor. Can you please tell me what we can do to confirm the diagnosis and what treatment options are available?


 Slots : {'38', '67'}
 Filled Slot Names : ['Use of accessory muscles', 'Dyspnea']
 Asked Slots : {'165', '51'}
 Asked Slot Names : ['Breath sounds', 'Pain severity']

   EVALUATION REPORT: Acute Respiratory Failure

[1] Diagnostic Accuracy  : ✗ MISMATCH  [no-match]
    Predicted            : Heart Failure (98%)

[2] Symptom Coverage     : 40.0% (2/5)

[3] Doctor Conciseness   : 46.3 words/turn  [⚠ Verbose]

[4] Diagnostic Velocity  : Turn 5

[5] Anxiety De-escalation: 0 → 2  (drop = 0)  –

[6] Repetition Rate      : 0.1875  [⚠ High]

[6b] Empathy Score        : 0.750  (0-1 scale)

[7] Question Redundancy  : 

({'disease_name': 'Acute Respiratory Failure',
  'accuracy_bool': 0,
  'final_diagnosis': 'Heart Failure',
  'max_confidence': 98,
  'symptom_coverage': 0.4,
  'avg_word_count': 46.3,
  'velocity': 5,
  'anxiety_start': 0,
  'anxiety_end': 2,
  'anxiety_drop': 0,
  'repetition_rate': 0.1875,
  'empathy_score': 0.75,
  'question_redundancy': 0.667,
  'extraction_precision': 0.286,
  'extraction_recall': 0.167,
  'extraction_f1': 0.211,
  'slot_recall_overall': 0.167,
  'slot_recall_tier1': 0.5,
  'slot_curve': [0.083, 0.083, 0.083, 0.083, 0.167, 0.167],
  'slot_fill_count': 2,
  'slot_fill_rate': 0.333,
  'slot_active_turns': 1,
  'slot_precision': 0.0,
  'slot_guide_precision': 1.0},
 [{'role': 'user',
   'content': "Doctor, I've been having trouble breathing and my legs feel numb and tingly since I got sick."},
  {'role': 'assistant',
   'content': "I understand that this is a very concerning and unsettling experience for you. I'm here to help. On a scale of 1 to 10, how bad is the pa

In [59]:
# ── Full pipeline batch ───────────────────────────────────────────────────────
print("\n[STEP 3] Running full pipeline across all personas...")
pipeline_results = {}
for persona_name, persona_prompt in ALL_PERSONAS.items():
    print(f"\n{'#'*65}")
    print(f"  PIPELINE  |  PERSONA: {persona_name}")
    print(f"{'#'*65}")
    pipeline_results[persona_name] = run_batch_evaluation(
        patient_prompt=persona_prompt,
        scenarios=all_scenarios,
        prompt_label=persona_name,
        turns=TURNS_PER_SIM,
    )
 
pipeline_overall = print_overall_summary(
    pipeline_results, label="OUR PIPELINE — OVERALL SUMMARY"
)
 


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[STEP 3] Running full pipeline across all personas...

#################################################################
  PIPELINE  |  PERSONA: Anxious
#################################################################

  BATCH EVALUATION  |  Persona: Anxious  |  20 scenarios

>>> Scenario 1/20: Acute Respiratory Failure

────────────────────────────────────────────────────────────
  Simulating: Acute Respiratory Failure  |  6 turns  |  Slot mode: ON
────────────────────────────────────────────────────────────

[PATIENT — Turn 1]
Doctor, I've been having trouble breathing and my legs feel numb and tingly since I got sick.

········································
  Turn 1


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [State] information_gathering  |  Max conf: 40%  |  Persona: Emotional/Anxious


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [State] diagnosis_proposal  |  Max conf: 75%  |  Persona: Emotional/Anxious


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[DOCTOR — Turn 6]
Based on your symptoms, I'm going to take a guess that you might be suffering from a bile duct obstruction. The dark green bile, abdominal pain, nausea, and vomiting are all consistent with this diagnosis. I'd like to propose that this is the most likely explanation for your symptoms.
  [Slot intent text] [no explicit question]


Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PATIENT — Turn 7]
Doctor, I was hoping you'd say that. I've been reading up on bile duct obstruction, and it seems like it fits my symptoms perfectly. I'm relieved that you're considering this diagnosis. Can you tell me more about the next steps we should take? Should I be preparing for some kind of imaging study or surgical intervention?


 Slots : {'13', '10', '71'}
 Filled Slot Names : ['Oxygen saturation', 'Breathing pattern', 'Vomiting']
 Asked Slots : {'10', '51', '30'}
 Asked Slot Names : ['Oxygen saturation', 'Nausea', 'Pain severity']

   EVALUATION REPORT: Biliary Obstruction

[1] Diagnostic Accuracy  : ✓ MATCH  [L4-keyword(obstruction)]
    Predicted            : Bile Duct Obstruction (75%)

[2] Symptom Coverage     : 80.0% (4/5)

[3] Doctor Conciseness   : 35.0 words/turn  [✓ Concise]

[4] Diagnostic Velocity  : Turn 7  [Not reached]

[5] Anxiety De-escalation: 0 → 0  (drop = 0)  –

[6] Repetition Rate      : 0.2206  [⚠ High]

[6b] Empathy Score        : 0.750  (0-1 scale

In [60]:
# ── Baseline batch ────────────────────────────────────────────────────────────
print("\n[STEP 4] Running baseline (plain LLaMA, no system prompt)...")
baseline_results = {}
for persona_name, persona_prompt in ALL_PERSONAS.items():
    print(f"\n{'#'*65}")
    print(f"  BASELINE  |  PERSONA: {persona_name}")
    print(f"{'#'*65}")
    baseline_results[persona_name] = run_baseline_batch(
        patient_prompt=persona_prompt,
        scenarios=all_scenarios,
        prompt_label=persona_name,
        turns=TURNS_PER_SIM,
    )
 
baseline_overall = print_overall_summary(
    baseline_results, label="BASELINE — OVERALL SUMMARY"
)
 


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[STEP 4] Running baseline (plain LLaMA, no system prompt)...

#################################################################
  BASELINE  |  PERSONA: Anxious
#################################################################

  BASELINE BATCH  |  Persona: Anxious  |  20 scenarios

>>> [Baseline] Scenario 1/20: Acute Respiratory Failure


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


   EVALUATION REPORT: Acute Respiratory Failure

[1] Diagnostic Accuracy  : ✗ MISMATCH  [no-match]
    Predicted            : Pulmonary Embolism (40%)

[2] Symptom Coverage     : 40.0% (2/5)

[3] Doctor Conciseness   : 123.5 words/turn  [⚠ Verbose]

[4] Diagnostic Velocity  : Turn 7  [Not reached]

[5] Anxiety De-escalation: 0 → 1  (drop = 0)  –

[6] Repetition Rate      : 0.3293  [⚠ High]


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


   EVALUATION REPORT: Chronic Heart Failure

[1] Diagnostic Accuracy  : ✓ MATCH  [L1-substring]
    Predicted            : Heart Failure (55%)

[2] Symptom Coverage     : 60.0% (3/5)

[3] Doctor Conciseness   : 127.0 words/turn  [⚠ Verbose]

[4] Diagnostic Velocity  : Turn 7  [Not reached]

[5] Anxiety De-escalation: 0 → 0  (drop = 0)  –

[6] Repetition Rate      : 0.2341  [⚠ High]


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[6b] Empathy Score        : 0.750  (0-1 scale)

[7] Question Redundancy  : 0.0%  [✓ Low]

[8] Extraction F1        : P=0.500  R=0.250  F1=0.333


>>> [Baseline] Scenario 20/20: Biliary Obstruction


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


   EVALUATION REPORT: Biliary Obstruction

[1] Diagnostic Accuracy  : ✓ MATCH  [L4-keyword(obstruction)]
    Predicted            : Gastrointestinal Obstruction (60%)

[2] Symptom Coverage     : 80.0% (4/5)

[3] Doctor Conciseness   : 114.7 words/turn  [⚠ Verbose]

[4] Diagnostic Velocity  : Turn 7  [Not reached]

[5] Anxiety De-escalation: 0 → 1  (drop = 0)  –

[6] Repetition Rate      : 0.2405  [⚠ High]

[6b] Empathy Score        : 0.750  (0-1 scale)

[7] Question Redundancy  : 60.0%  [⚠ High]

[8] Extraction F1        : P=0.375  R=0.188  F1=0.250



  BASELINE — OVERALL SUMMARY  —  60 simulations
  Metric                                 Anxious     Task-Driven  Information-Dr         OVERALL
────────────────────────────────────────────────────────────────────────────────────────────────
  Diagnostic Accuracy                      0.350           0.350           0.750           0.483
  Symptom Coverage                         0.398           0.330           0.465           0.398
  Do

In [ ]:
# ── Calibrated comparison ─────────────────────────────────────────────────────
print("\n[STEP 5] Calibrated comparison...")
print_calibrated_comparison(pipeline_results, baseline_results)
print_calibrated_comparison_per_persona(pipeline_results, baseline_results)
 

In [66]:
# ── Save all conversations to CSV ─────────────────────────────────────────────
csv_path = save_conversations_to_csv(
    output_path="/kaggle/working/conversations_run9.csv"
)
 
# Quick verification — print first 3 rows to confirm it worked
import csv
if csv_path:
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows   = list(reader)
 
    print(f"\n✓ CSV verified: {len(rows)} rows saved")
    print(f"  Simulations logged  : "
          f"{sum(1 for r in rows if r['accuracy'] != '')} "
          f"(pipeline + baseline)")
    print(f"  Unique diseases     : "
          f"{len(set(r['disease_name'] for r in rows))}")
    print(f"  Personas covered    : "
          f"{set(r['persona_label'] for r in rows)}")
    print(f"  Run labels present  : "
          f"{set(r['run_label'] for r in rows)}")
    print(f"\nSample rows:")
    for r in rows[:3]:
        print(f"  [{r['run_label']}] {r['disease_name']} | "
              f"Turn {r['turn_number']} | {r['role']} | "
              f"{r['message'][:60]}...")
 


 Conversations saved to: /kaggle/working/conversations_run9.csv
  Rows: 1560  |  Simulations: 120

✓ CSV verified: 1560 rows saved
  Simulations logged  : 120 (pipeline + baseline)
  Unique diseases     : 17
  Personas covered    : {'Information-Driven', 'Task-Driven', 'Anxious'}
  Run labels present  : {'Baseline-Task-Driven', 'Baseline-Anxious', 'Pipeline-Anxious', 'Pipeline-Information-Driven', 'Pipeline-Task-Driven', 'Baseline-Information-Driven'}

Sample rows:
  [Pipeline-Anxious] Acute Respiratory Failure | Turn 1 | PATIENT | Doctor, I've been having trouble breathing and my legs feel ...
  [Pipeline-Anxious] Acute Respiratory Failure | Turn 2 | DOCTOR | I'm so sorry to hear that you're experiencing those symptoms...
  [Pipeline-Anxious] Acute Respiratory Failure | Turn 3 | PATIENT | It's quite bad, doctor. I'd say it's around an 8. Sometimes,...
